# ASS Inpaint Mesh — Static & SAM 2 Moving-Object GPU Video Inpainting for Aegisub (G4 / L4 / T4 GPU)

Erase on-screen text, signs, **or moving/deforming objects** over complex backgrounds (drifting clouds, flowing water, camera pans, moving props) on Google Colab (**G4 `RTX PRO 6000 Blackwell 96GB`**, **A100/L4**, or **T4 GPU**), and export optimized frame-by-frame **ASS `\p1` vector drawing lines** ready to import or paste into Aegisub.

### Two Masking Modes (Checkbox Selector in Section 2B)
- **Mask static object**: Paste your existing Aegisub `\clip(m ...)` polygon (optionally with rigid Lucas-Kanade optical-flow tracking for simple camera pans).
- **Mask moving object (SAM 2 Video Predictor)**: Click or box-select any moving/deforming object on Frame 0 using **Meta's SAM 2.1 (`sam2.1_hiera_large`)**, track its exact silhouette across every frame, auto-generate frame-by-frame ASS `\clip(...)` masks, inpaint the background behind it, and rebuild as optimized `\p1` ASS meshes.

### Pipeline Overview
1. **Environment & GPU Auto-Configuration**: Detects G4 / L4 / T4 VRAM, configures `bfloat16`/`float16`, and sets up `ProPainter`, `DiffuEraser`, `SimpleLama`, and `SAM 2.1`.
2. **Video & Mask Mode Setup**: Upload a clip or mount Google Drive, pick **Mask static object** vs. **Mask moving object (SAM 2 Video Predictor)** via the 2-option checkboxes, and configure timing (`HH:MM:SS.cs`).
3. **Interactive Object Selection (3A) & Mask Preview (3B)**: Click/box-select the moving object on Frame 0 (in SAM 2 mode) and preview tracked mask + crop box coverage across First / Middle / Last frames.
4. **Temporally Consistent GPU Video Inpainting**: Crops around the padded union bounding box, runs `DiffuEraser`, `ProPainter + LaMa`, `ProPainter`, or `Optical-Flow Fallback`, applies boundary color harmonization, and shows a side-by-side preview.
5. **Frame-by-Frame ASS Vector Mesh Generator**: Applies **Greedy 2D Rectangle Merging** and color grouping (`\1c&HBBGGRR&`) per frame/keyframe step to prevent `libass` playback lag.
6. **Export & Download**: Writes `rebuilt_motion_patch.ass` with full `[Script Info]` and `[V4+ Styles]` headers and triggers download.

## 1. Environment & G4 / T4 GPU Setup

Select your hardware runtime in Colab (**Runtime → Change runtime type → `G4 GPU`**, `L4 GPU`, or `T4 GPU`).
This cell automatically detects your GPU VRAM tier and installs **ProPainter**, **DiffuEraser** (for G4/L4 SOTA diffusion quality), **SimpleLama**, and **SAM 2.1 (`sam2.1_hiera_large`)** for moving-object tracking.

In [ ]:
#@title 1. Auto-Detect GPU (G4 / L4 / T4), Install Dependencies & Setup Models { display-mode: "form" }
import os
import shutil
import subprocess
import sys
import urllib.request

SETUP_DIFFUERASER = True  #@param {type:"boolean"}  # Recommended on G4 / A100 / L4 GPUs
SETUP_LAMA_REFINER = True  #@param {type:"boolean"}  # Used by ProPainter + LaMa hybrid engine
SETUP_SAM2 = True  #@param {type:"boolean"}  # Required for 'Mask moving object (SAM 2 Video Predictor)'

# 1. Detect GPU & VRAM tier to set optimal defaults for Cell 3 & Cell 4
GPU_NAME = "CPU"
VRAM_GB = 0.0
HAS_CUDA = False
SAM2_AUTOCAST_DTYPE_NAME = "float16"
OFFLOAD_VIDEO_TO_CPU = True

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    if HAS_CUDA:
        props = torch.cuda.get_device_properties(0)
        GPU_NAME = props.name
        VRAM_GB = props.total_memory / (1024 ** 3)
        if torch.cuda.is_bf16_supported() and props.major >= 8:
            SAM2_AUTOCAST_DTYPE_NAME = "bfloat16"
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        else:
            SAM2_AUTOCAST_DTYPE_NAME = "float16"
        OFFLOAD_VIDEO_TO_CPU = VRAM_GB <= 18.0
        print(f"CUDA available: {GPU_NAME} ({VRAM_GB:.1f} GB VRAM, SM {props.major}.{props.minor})")
    else:
        print("WARNING: CUDA GPU not detected! Enable Runtime -> Change runtime type -> G4 GPU or T4 GPU.")
except ImportError:
    print("PyTorch not found yet; checking nvidia-smi...")

subprocess.run(["nvidia-smi"], check=False)

if VRAM_GB >= 40.0:
    GPU_TIER = "G4 / High-VRAM (>=40 GB)"
    RECOMMENDED_PROFILE = {
        "subvideo_length": 120,
        "neighbor_length": 20,
        "ref_stride": 5,
        "use_fp16": False,  # FP32 prevents subtle banding on smooth gradients
        "max_img_size": 1920,
        "diffueraser_steps": 4,
    }
elif VRAM_GB >= 22.0:
    GPU_TIER = "L4 / Mid-VRAM (22-39 GB)"
    RECOMMENDED_PROFILE = {
        "subvideo_length": 80,
        "neighbor_length": 15,
        "ref_stride": 8,
        "use_fp16": True,
        "max_img_size": 1280,
        "diffueraser_steps": 2,
    }
else:
    GPU_TIER = "T4 / Standard (<=21 GB)" if HAS_CUDA else "CPU"
    RECOMMENDED_PROFILE = {
        "subvideo_length": 50,
        "neighbor_length": 10,
        "ref_stride": 10,
        "use_fp16": True,
        "max_img_size": 960,
        "diffueraser_steps": 2,
    }

print(f"Detected Tier : {GPU_TIER}")
print(f"Auto Profile  : {RECOMMENDED_PROFILE}")
print(f"SAM 2 Config  : autocast={SAM2_AUTOCAST_DTYPE_NAME}, offload_video_to_cpu={OFFLOAD_VIDEO_TO_CPU}")

# 2. Install packages (without overwriting Colab's CUDA/Blackwell PyTorch build)
REQUIRED_PKGS = [
    "opencv-python>=4.8.0",
    "numpy>=1.24.0",
    "scipy>=1.10.0",
    "ipywidgets>=8.0.0",
    "matplotlib>=3.7.0",
    "Pillow>=10.0.0",
    "scikit-image>=0.21.0",
    "einops>=0.6.1",
    "timm>=0.9.0",
    "av>=10.0.0",
    "imageio>=2.31.0",
    "imageio-ffmpeg>=0.4.8",
    "omegaconf>=2.3.0",
]
if SETUP_DIFFUERASER:
    REQUIRED_PKGS.extend([
        "diffusers>=0.29.0",
        "transformers>=4.40.0",
        "accelerate>=0.30.0",
        "huggingface_hub>=0.23.0",
        "peft>=0.11.0",
        "safetensors>=0.4.3",
    ])
if SETUP_LAMA_REFINER:
    REQUIRED_PKGS.append("simple-lama-inpainting>=0.1.0")
if SETUP_SAM2:
    REQUIRED_PKGS.extend([
        "ipympl>=0.9.3",
        "git+https://github.com/facebookresearch/sam2.git",
    ])

print("\nInstalling Python packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PKGS])

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

# Fix Matplotlib 3.9+/3.10+ entry-point cache when ipympl is installed mid-session
try:
    from matplotlib.backends import backend_registry
    for _bname in ("widget", "ipympl"):
        backend_registry._name_to_module[_bname] = "module://ipympl.backend_nbagg"
        backend_registry._backend_to_gui_framework[_bname] = "unknown"
except Exception:
    pass

# 3. Clone sczhou/ProPainter and download pretrained weights
CONTENT_ROOT = "/content" if os.path.exists("/content") else os.getcwd()
PROPAINTER_DIR = os.path.join(CONTENT_ROOT, "ProPainter")
WEIGHTS_DIR = os.path.join(PROPAINTER_DIR, "weights")

if not os.path.exists(PROPAINTER_DIR):
    print(f"\nCloning sczhou/ProPainter into {PROPAINTER_DIR}...")
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/sczhou/ProPainter.git", PROPAINTER_DIR])
else:
    print(f"\nProPainter repo already present at {PROPAINTER_DIR}")

os.makedirs(WEIGHTS_DIR, exist_ok=True)
PROPAINTER_WEIGHTS = {
    "ProPainter.pth": "https://github.com/sczhou/ProPainter/releases/download/v0.1.0/ProPainter.pth",
    "recurrent_flow_completion.pth": "https://github.com/sczhou/ProPainter/releases/download/v0.1.0/recurrent_flow_completion.pth",
    "raft-things.pth": "https://github.com/sczhou/ProPainter/releases/download/v0.1.0/raft-things.pth",
    "i3d_rgb_imagenet.pt": "https://github.com/sczhou/ProPainter/releases/download/v0.1.0/i3d_rgb_imagenet.pt",
}

for fname, url in PROPAINTER_WEIGHTS.items():
    dst = os.path.join(WEIGHTS_DIR, fname)
    if os.path.exists(dst) and os.path.getsize(dst) > 1024 * 1024:
        print(f"  [ok] ProPainter/{fname} already downloaded")
        continue
    print(f"  Downloading ProPainter/{fname}...")
    try:
        urllib.request.urlretrieve(url, dst)
        print(f"  [ok] ProPainter/{fname} ({os.path.getsize(dst) / (1024 ** 2):.1f} MB)")
    except Exception as exc:
        print(f"  [warn] Failed to download {fname}: {exc}")

# 4. Clone lixiaowen-xw/DiffuEraser and download HuggingFace weights (recommended on G4 GPU)
DIFFUERASER_DIR = os.path.join(CONTENT_ROOT, "DiffuEraser")
if SETUP_DIFFUERASER:
    if not os.path.exists(DIFFUERASER_DIR):
        print(f"\nCloning lixiaowen-xw/DiffuEraser into {DIFFUERASER_DIR}...")
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/lixiaowen-xw/DiffuEraser.git", DIFFUERASER_DIR])
    else:
        print(f"\nDiffuEraser repo already present at {DIFFUERASER_DIR}")

    de_weights = os.path.join(DIFFUERASER_DIR, "weights")
    os.makedirs(de_weights, exist_ok=True)
    de_prop_dir = os.path.join(de_weights, "propainter")
    os.makedirs(de_prop_dir, exist_ok=True)
    for fname in ("ProPainter.pth", "recurrent_flow_completion.pth", "raft-things.pth"):
        src_w = os.path.join(WEIGHTS_DIR, fname)
        dst_w = os.path.join(de_prop_dir, fname)
        if os.path.exists(src_w) and not os.path.exists(dst_w):
            try:
                os.symlink(src_w, dst_w)
            except OSError:
                shutil.copy2(src_w, dst_w)

    from huggingface_hub import snapshot_download
    HF_DOWNLOADS = [
        ("lixiaowen/diffuEraser", os.path.join(de_weights, "diffuEraser"), ["brushnet/*", "unet_main/*"]),
        ("stable-diffusion-v1-5/stable-diffusion-v1-5", os.path.join(de_weights, "stable-diffusion-v1-5"),
         ["feature_extractor/*", "model_index.json", "safety_checker/*", "scheduler/*", "text_encoder/*", "tokenizer/*"]),
        ("wangfuyun/PCM_Weights", os.path.join(de_weights, "PCM_Weights"), ["sd15/*"]),
        ("stabilityai/sd-vae-ft-mse", os.path.join(de_weights, "sd-vae-ft-mse"), None),
    ]
    for repo_id, local_dir, patterns in HF_DOWNLOADS:
        if os.path.exists(local_dir) and len(os.listdir(local_dir)) > 0:
            print(f"  [ok] {repo_id} already present")
            continue
        print(f"  Downloading {repo_id}...")
        try:
            snapshot_download(repo_id=repo_id, local_dir=local_dir, allow_patterns=patterns)
            print(f"  [ok] {repo_id}")
        except Exception as exc:
            print(f"  [warn] Failed to download {repo_id}: {exc}")

# 5. Download SAM 2.1 Video Predictor checkpoint
SAM2_CKPT_DIR = os.path.join(CONTENT_ROOT, "checkpoints")
SAM2_CKPT_PATH = os.path.join(SAM2_CKPT_DIR, "sam2.1_hiera_large.pt")
SAM2_CKPT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
if SETUP_SAM2:
    os.makedirs(SAM2_CKPT_DIR, exist_ok=True)
    if os.path.exists(SAM2_CKPT_PATH) and os.path.getsize(SAM2_CKPT_PATH) > 10 * 1024 * 1024:
        print(f"  [ok] SAM 2.1 checkpoint already downloaded ({SAM2_CKPT_PATH})")
    else:
        print(f"\nDownloading SAM 2.1 checkpoint ({os.path.basename(SAM2_CKPT_PATH)})...")
        try:
            urllib.request.urlretrieve(SAM2_CKPT_URL, SAM2_CKPT_PATH)
            print(f"  [ok] SAM 2.1 checkpoint ({os.path.getsize(SAM2_CKPT_PATH) / (1024 ** 2):.1f} MB)")
        except Exception as exc:
            print(f"  [warn] Failed to download SAM 2.1 checkpoint: {exc}")

print("\nSetup complete.")

## 2. Video & Mask Mode Setup

1. Run **Cell 2A** to upload a trimmed clip (`.mp4`/`.mkv`) from your computer or load one from Google Drive.
2. Run **Cell 2B** to choose between the two masking modes via the checkboxes:
   - **Mask static object**: Paste your Aegisub `\clip(m ...)` vector string.
   - **Mask moving object (SAM 2 Video Predictor)**: Track any moving/deforming object interactively in Section 3A.

In [ ]:
#@title 2A. Load Video Clip (Upload or Google Drive) { display-mode: "form" }
import os
import cv2

SOURCE_MODE = "Upload File (Colab Browser Upload)"  #@param ["Upload File (Colab Browser Upload)", "Google Drive Path", "Local / Existing Runtime Path"]
DRIVE_OR_LOCAL_PATH = "/content/drive/MyDrive/clip.mp4"  #@param {type:"string"}

VIDEO_PATH = None

if SOURCE_MODE == "Upload File (Colab Browser Upload)":
    try:
        from google.colab import files
        print("Select your trimmed video clip (.mp4 / .mkv / .mov)...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file was uploaded.")
        first_name = next(iter(uploaded.keys()))
        VIDEO_PATH = os.path.abspath(first_name)
    except ImportError:
        raise RuntimeError("google.colab is not available here. Switch SOURCE_MODE to 'Local / Existing Runtime Path'.")
elif SOURCE_MODE == "Google Drive Path":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass
    VIDEO_PATH = DRIVE_OR_LOCAL_PATH.strip()
else:
    VIDEO_PATH = DRIVE_OR_LOCAL_PATH.strip()

if not VIDEO_PATH or not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Video file not found: {VIDEO_PATH!r}")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"OpenCV failed to open video: {VIDEO_PATH!r}")

VIDEO_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VIDEO_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
VIDEO_FPS = float(cap.get(cv2.CAP_PROP_FPS) or 23.976)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

duration_s = TOTAL_FRAMES / VIDEO_FPS if VIDEO_FPS > 0 else 0.0
print(f"Loaded Video: {VIDEO_PATH}")
print(f"Resolution  : {VIDEO_WIDTH} x {VIDEO_HEIGHT}")
print(f"FPS         : {VIDEO_FPS:.4f}")
print(f"Frame Count : {TOTAL_FRAMES} ({duration_s:.2f} s)")

In [ ]:
#@title 2B. Choose Object Mask Mode & Configure Parameters { display-mode: "form" }
import math
import re
from typing import Dict, List, Optional, Sequence, Tuple
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Choose mode via the 2-option checkboxes below (or Colab dropdown)
OBJECT_MASK_MODE = "Mask static object"  #@param ["Mask static object", "Mask moving object (SAM 2 Video Predictor)"]

# Common frame & padding settings
START_FRAME = 0  #@param {type:"integer"}
END_FRAME = -1  #@param {type:"integer"}  # -1 = process until last frame
START_TIME_OFFSET = "0:00:00.00"  #@param {type:"string"}  # Episode timestamp of Frame 0 (H:MM:SS.cs)
PLAY_RES_X = 0  #@param {type:"integer"}  # 0 = match video width
PLAY_RES_Y = 0  #@param {type:"integer"}  # 0 = match video height
MASK_DILATION_PX = 4  #@param {type:"slider", min:0, max:20, step:1}
CROP_PADDING_PX = 64  #@param {type:"slider", min:16, max:256, step:8}

# Static object settings (used when 'Mask static object' is checked)
ASS_CLIP_STRING = "\\clip(m 400 200 l 800 200 800 300 400 300)"  #@param {type:"string"}
STATIC_RIGID_LK_TRACK = False  #@param {type:"boolean"}  # Optional rigid optical-flow pan tracking for static signs

# Moving object (SAM 2) settings (used when 'Mask moving object' is checked)
SAM2_CLIP_EXPAND_PX = 2  #@param {type:"slider", min:0, max:12, step:1}
SAM2_CLIP_EPSILON = 1.5  #@param {type:"slider", min:0.5, max:8.0, step:0.5}
SAM2_MIN_CONTOUR_AREA = 16.0  #@param {type:"number"}


def parse_ass_time_to_ms(ts: str) -> int:
    """Parse ASS timestamp 'H:MM:SS.cs' (or 'MM:SS.cs') into integer milliseconds."""
    s = (ts or "0:00:00.00").strip()
    m = re.match(r"^(?:(\d+):)?(\d{1,2}):(\d{2})(?:\.(\d{1,3}))?$", s)
    if not m:
        raise ValueError(f"Invalid ASS timestamp {ts!r}. Expected H:MM:SS.cs (e.g. 0:04:12.50).")
    h = int(m.group(1) or 0)
    mins = int(m.group(2))
    secs = int(m.group(3))
    frac = m.group(4) or "0"
    if len(frac) == 1:
        ms = int(frac) * 100
    elif len(frac) == 2:
        ms = int(frac) * 10
    else:
        ms = int(frac[:3])
    return ((h * 3600 + mins * 60 + secs) * 1000) + ms


def ms_to_ass_time(ms: int) -> str:
    """Format milliseconds into ASS timestamp 'H:MM:SS.cs'."""
    ms = max(0, int(ms))
    cs_total = round(ms / 10.0)
    cs = cs_total % 100
    s_total = cs_total // 100
    s = s_total % 60
    m_total = s_total // 60
    m = m_total % 60
    h = m_total // 60
    return f"{h}:{m:02d}:{s:02d}.{cs:02d}"


def extract_raw_clip_inner(clip_str: str) -> str:
    s = (clip_str or "").strip()
    m = re.match(r"^\\?i?clip\((.*)\)$", s, flags=re.IGNORECASE | re.DOTALL)
    return m.group(1).strip() if m else s


def _eval_cubic_bezier(
    p0: Tuple[float, float],
    p1: Tuple[float, float],
    p2: Tuple[float, float],
    p3: Tuple[float, float],
) -> List[Tuple[float, float]]:
    chord = (
        math.hypot(p1[0] - p0[0], p1[1] - p0[1])
        + math.hypot(p2[0] - p1[0], p2[1] - p1[1])
        + math.hypot(p3[0] - p2[0], p3[1] - p2[1])
    )
    steps = max(12, min(64, int(math.ceil(chord / 2.0))))
    pts: List[Tuple[float, float]] = []
    for i in range(1, steps + 1):
        t = i / steps
        mt = 1.0 - t
        mt2 = mt * mt
        t2 = t * t
        x = (mt2 * mt) * p0[0] + (3 * mt2 * t) * p1[0] + (3 * mt * t2) * p2[0] + (t2 * t) * p3[0]
        y = (mt2 * mt) * p0[1] + (3 * mt2 * t) * p1[1] + (3 * mt * t2) * p2[1] + (t2 * t) * p3[1]
        pts.append((x, y))
    return pts


def _eval_uniform_bspline(ctrl_pts: Sequence[Tuple[float, float]], closed: bool = False) -> List[Tuple[float, float]]:
    if len(ctrl_pts) < 4:
        return list(ctrl_pts[1:])
    pts = list(ctrl_pts)
    pts = (pts + pts[:3]) if closed else ([pts[0], pts[0]] + pts + [pts[-1], pts[-1]])
    out: List[Tuple[float, float]] = []
    for i in range(len(pts) - 3):
        p0, p1, p2, p3 = pts[i], pts[i + 1], pts[i + 2], pts[i + 3]
        seg_len = (
            math.hypot(p1[0] - p0[0], p1[1] - p0[1])
            + math.hypot(p2[0] - p1[0], p2[1] - p1[1])
            + math.hypot(p3[0] - p2[0], p3[1] - p2[1])
        )
        steps = max(8, min(32, int(math.ceil(seg_len / 2.0))))
        for s in range(1, steps + 1):
            t = s / steps
            t2 = t * t
            t3 = t2 * t
            b0 = (-t3 + 3 * t2 - 3 * t + 1) / 6.0
            b1 = (3 * t3 - 6 * t2 + 4) / 6.0
            b2 = (-3 * t3 + 3 * t2 + 3 * t + 1) / 6.0
            b3 = t3 / 6.0
            out.append((b0 * p0[0] + b1 * p1[0] + b2 * p2[0] + b3 * p3[0], b0 * p0[1] + b1 * p1[1] + b2 * p2[1] + b3 * p3[1]))
    return out


def parse_ass_clip_structured(clip_str: str) -> Tuple[List[List[Tuple[str, List[Tuple[float, float]]]]], List[List[Tuple[float, float]]], str]:
    """
    Parse ASS \\clip(...) into both:
    - structured commands per contour: [('m', [(x,y)]), ('l', [(x,y)]), ('b', [p1,p2,p3]), ...]
    - subdivided polygon contours: [[(x,y), ...], ...]
    - normalized raw_clip_inner string
    """
    raw_inner = extract_raw_clip_inner(clip_str)
    if not raw_inner:
        raise ValueError("Empty ASS \\clip string.")

    scale_factor = 1.0
    drawing_str = raw_inner
    scale_match = re.match(r"^\s*(\d+)\s*,\s*(.+)$", raw_inner, flags=re.DOTALL)
    if scale_match:
        maybe_scale = int(scale_match.group(1))
        rest = scale_match.group(2).strip()
        if re.search(r"[mnlbspc]", rest, flags=re.IGNORECASE):
            if maybe_scale >= 1:
                scale_factor = float(1 << (maybe_scale - 1))
            drawing_str = rest

    if not re.search(r"[a-zA-Z]", drawing_str):
        nums = re.findall(r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?", drawing_str)
        if len(nums) == 4:
            x1, y1, x2, y2 = (float(v) for v in nums)
            xmin, xmax = min(x1, x2), max(x1, x2)
            ymin, ymax = min(y1, y2), max(y1, y2)
            poly = [(xmin, ymin), (xmax, ymin), (xmax, ymax), (xmin, ymax)]
            cmds = [[("m", [(xmin, ymin)]), ("l", [(xmax, ymin)]), ("l", [(xmax, ymax)]), ("l", [(xmin, ymax)])]]
            return cmds, [poly], f"m {xmin:g} {ymin:g} l {xmax:g} {ymin:g} {xmax:g} {ymax:g} {xmin:g} {ymax:g}"
        raise ValueError(f"Could not parse rectangular \\clip: {raw_inner!r}")

    token_re = re.compile(r"([mnlbspcMNLBSPC])|([-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?)")
    tokens = [("CMD", m.group(1).lower()) if m.group(1) else ("NUM", m.group(2)) for m in token_re.finditer(drawing_str)]

    all_cmds: List[List[Tuple[str, List[Tuple[float, float]]]]] = []
    cur_cmds: List[Tuple[str, List[Tuple[float, float]]]] = []
    contours: List[List[Tuple[float, float]]] = []
    cur_poly: List[Tuple[float, float]] = []
    spline_pts: List[Tuple[float, float]] = []
    in_spline = False
    cmd: Optional[str] = None
    idx = 0

    def flush_spline(closed: bool = False) -> None:
        nonlocal in_spline, spline_pts
        if in_spline and len(spline_pts) >= 2:
            cur_poly.extend(_eval_uniform_bspline(spline_pts, closed=closed))
        in_spline = False
        spline_pts = []

    def flush_contour() -> None:
        nonlocal cur_poly, cur_cmds
        flush_spline(closed=False)
        if len(cur_poly) >= 3:
            contours.append(cur_poly)
            all_cmds.append(cur_cmds)
        cur_poly = []
        cur_cmds = []

    def read_nums(n: int) -> Optional[List[float]]:
        nonlocal idx
        vals: List[float] = []
        scan = idx
        while scan < len(tokens) and tokens[scan][0] == "NUM" and len(vals) < n:
            vals.append(float(tokens[scan][1]) / scale_factor)
            scan += 1
        if len(vals) < n:
            return None
        idx = scan
        return vals

    while idx < len(tokens):
        ttype, tval = tokens[idx]
        if ttype == "CMD":
            if tval == "c":
                flush_spline(closed=True)
                cur_cmds.append(("c", []))
                idx += 1
                continue
            if tval != "p" and in_spline:
                flush_spline(closed=False)
            cmd = tval
            idx += 1
            continue

        if cmd is None:
            cmd = "m"

        if cmd in ("m", "n"):
            coords = read_nums(2)
            if coords is None:
                break
            flush_contour()
            pt = (coords[0], coords[1])
            cur_poly.append(pt)
            cur_cmds.append((cmd, [pt]))
            cmd = "l"
        elif cmd == "l":
            coords = read_nums(2)
            if coords is None:
                break
            pt = (coords[0], coords[1])
            cur_poly.append(pt)
            cur_cmds.append(("l", [pt]))
        elif cmd == "b":
            coords = read_nums(6)
            if coords is None:
                break
            p0 = cur_poly[-1] if cur_poly else (0.0, 0.0)
            p1, p2, p3 = (coords[0], coords[1]), (coords[2], coords[3]), (coords[4], coords[5])
            cur_poly.extend(_eval_cubic_bezier(p0, p1, p2, p3))
            cur_cmds.append(("b", [p1, p2, p3]))
        elif cmd in ("s", "p"):
            coords = read_nums(2)
            if coords is None:
                break
            pt = (coords[0], coords[1])
            if not in_spline:
                in_spline = True
                spline_pts = [cur_poly[-1] if cur_poly else pt]
            spline_pts.append(pt)
            cur_cmds.append((cmd, [pt]))
        else:
            idx += 1

    flush_contour()
    if not contours:
        raise ValueError(f"No valid contours parsed from \\clip: {raw_inner!r}")
    return all_cmds, contours, raw_inner


def format_commands_as_clip_inner(all_cmds: Sequence[Sequence[Tuple[str, List[Tuple[float, float]]]]]) -> str:
    """Serialize structured ASS commands back into compact 'm ... l ... b ...' clip inner text."""
    parts: List[str] = []
    for contour_cmds in all_cmds:
        prev_cmd = None
        for cmd_name, pts in contour_cmds:
            if cmd_name == "c":
                parts.append("c")
                prev_cmd = "c"
                continue
            coords_str = " ".join(f"{int(round(px))} {int(round(py))}" for px, py in pts)
            if cmd_name == prev_cmd and cmd_name in ("l", "b", "p"):
                parts.append(coords_str)
            else:
                parts.append(f"{cmd_name} {coords_str}")
                prev_cmd = cmd_name
    return " ".join(parts)


def eval_commands_to_polygons(all_cmds: Sequence[Sequence[Tuple[str, List[Tuple[float, float]]]]]) -> List[List[Tuple[float, float]]]:
    """Convert structured ASS commands (including cubic Beziers) into dense polygon vertices."""
    polys: List[List[Tuple[float, float]]] = []
    for contour_cmds in all_cmds:
        poly: List[Tuple[float, float]] = []
        spline_pts: List[Tuple[float, float]] = []
        in_spline = False
        for cmd_name, pts in contour_cmds:
            if cmd_name == "c":
                if in_spline and len(spline_pts) >= 2:
                    poly.extend(_eval_uniform_bspline(spline_pts, closed=True))
                in_spline = False
                spline_pts = []
                continue
            if cmd_name != "p" and in_spline:
                if len(spline_pts) >= 2:
                    poly.extend(_eval_uniform_bspline(spline_pts, closed=False))
                in_spline = False
                spline_pts = []
            if cmd_name in ("m", "n", "l"):
                poly.append(pts[0])
            elif cmd_name == "b":
                p0 = poly[-1] if poly else (0.0, 0.0)
                poly.extend(_eval_cubic_bezier(p0, pts[0], pts[1], pts[2]))
            elif cmd_name in ("s", "p"):
                if not in_spline:
                    in_spline = True
                    spline_pts = [poly[-1] if poly else pts[0]]
                spline_pts.append(pts[0])
        if in_spline and len(spline_pts) >= 2:
            poly.extend(_eval_uniform_bspline(spline_pts, closed=False))
        if len(poly) >= 3:
            polys.append(poly)
    return polys


def rasterize_clip_mask(
    contours: Sequence[Sequence[Tuple[float, float]]],
    vid_w: int,
    vid_h: int,
    script_w: int,
    script_h: int,
) -> np.ndarray:
    sx = float(vid_w) / float(script_w) if script_w > 0 else 1.0
    sy = float(vid_h) / float(script_h) if script_h > 0 else 1.0
    mask = np.zeros((vid_h, vid_w), dtype=np.uint8)
    poly_arrays = []
    for contour in contours:
        if len(contour) < 3:
            continue
        arr = np.asarray(contour, dtype=np.float64)
        arr[:, 0] *= sx
        arr[:, 1] *= sy
        poly_arrays.append(np.round(arr).astype(np.int32))
    if poly_arrays:
        cv2.fillPoly(mask, poly_arrays, 255)
    return mask


def _transform_cmds_affine(
    all_cmds: Sequence[Sequence[Tuple[str, List[Tuple[float, float]]]]],
    M_vid: np.ndarray,
    vid_w: int,
    vid_h: int,
    script_w: int,
    script_h: int,
) -> List[List[Tuple[str, List[Tuple[float, float]]]]]:
    """Apply a 2x3 affine matrix defined in video pixel space to script-space ASS commands."""
    sx_to_vid = float(vid_w) / float(script_w) if script_w > 0 else 1.0
    sy_to_vid = float(vid_h) / float(script_h) if script_h > 0 else 1.0
    sx_to_scr = 1.0 / sx_to_vid
    sy_to_scr = 1.0 / sy_to_vid

    out_cmds: List[List[Tuple[str, List[Tuple[float, float]]]]] = []
    for contour_cmds in all_cmds:
        new_contour = []
        for cmd_name, pts in contour_cmds:
            if not pts:
                new_contour.append((cmd_name, []))
                continue
            new_pts = []
            for x, y in pts:
                vx, vy = x * sx_to_vid, y * sy_to_vid
                nx = M_vid[0, 0] * vx + M_vid[0, 1] * vy + M_vid[0, 2]
                ny = M_vid[1, 0] * vx + M_vid[1, 1] * vy + M_vid[1, 2]
                new_pts.append((nx * sx_to_scr, ny * sy_to_scr))
            new_contour.append((cmd_name, new_pts))
        out_cmds.append(new_contour)
    return out_cmds


def track_clip_contours_lk(
    frames_bgr: Sequence[np.ndarray],
    base_cmds: Sequence[Sequence[Tuple[str, List[Tuple[float, float]]]]],
    raw_clip_inner: str,
    vid_w: int,
    vid_h: int,
    script_w: int,
    script_h: int,
) -> Tuple[List[List[List[Tuple[float, float]]]], List[str]]:
    """
    Track \\clip contours across frames using Lucas-Kanade Optical Flow
    with forward-backward filtering and RANSAC partial-affine estimation.
    """
    n_frames = len(frames_bgr)
    base_polys = eval_commands_to_polygons(base_cmds)
    if n_frames <= 1:
        return [base_polys], [raw_clip_inner]

    cur_cmds = [list(c) for c in base_cmds]
    tracked_polys: List[List[List[Tuple[float, float]]]] = [base_polys]
    tracked_clips: List[str] = [raw_clip_inner]

    lk_params = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    prev_gray = cv2.cvtColor(frames_bgr[0], cv2.COLOR_BGR2GRAY)

    def detect_features(gray_img: np.ndarray, polys: Sequence[Sequence[Tuple[float, float]]]) -> Optional[np.ndarray]:
        m_in = rasterize_clip_mask(polys, vid_w, vid_h, script_w, script_h)
        m_dil = cv2.dilate(m_in, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
        pts = cv2.goodFeaturesToTrack(gray_img, maxCorners=120, qualityLevel=0.01, minDistance=5, mask=m_dil)
        if pts is None or len(pts) < 12:
            m_ring = cv2.dilate(m_in, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (65, 65)))
            pts = cv2.goodFeaturesToTrack(gray_img, maxCorners=160, qualityLevel=0.01, minDistance=6, mask=m_ring)
        return pts

    prev_pts = detect_features(prev_gray, base_polys)

    for i in range(1, n_frames):
        curr_gray = cv2.cvtColor(frames_bgr[i], cv2.COLOR_BGR2GRAY)
        M_step = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], dtype=np.float64)

        if prev_pts is not None and len(prev_pts) >= 4:
            next_pts, st_f, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None, **lk_params)
            back_pts, st_b, _ = cv2.calcOpticalFlowPyrLK(curr_gray, prev_gray, next_pts, None, **lk_params)
            if next_pts is not None and back_pts is not None:
                fb_dist = np.linalg.norm(prev_pts.reshape(-1, 2) - back_pts.reshape(-1, 2), axis=1)
                valid = (st_f.reshape(-1) == 1) & (st_b.reshape(-1) == 1) & (fb_dist < 1.5)
                p0 = prev_pts.reshape(-1, 2)[valid]
                p1 = next_pts.reshape(-1, 2)[valid]

                if len(p0) >= 6:
                    M_est, _ = cv2.estimateAffinePartial2D(p0, p1, method=cv2.RANSAC, ransacReprojThreshold=2.5)
                    if M_est is not None:
                        scale = math.hypot(M_est[0, 0], M_est[1, 0])
                        if 0.96 <= scale <= 1.04:
                            M_step = M_est
                        else:
                            delta = np.median(p1 - p0, axis=0)
                            M_step[0, 2], M_step[1, 2] = float(delta[0]), float(delta[1])
                    else:
                        delta = np.median(p1 - p0, axis=0)
                        M_step[0, 2], M_step[1, 2] = float(delta[0]), float(delta[1])
                elif len(p0) >= 1:
                    delta = np.median(p1 - p0, axis=0)
                    M_step[0, 2], M_step[1, 2] = float(delta[0]), float(delta[1])

        cur_cmds = _transform_cmds_affine(cur_cmds, M_step, vid_w, vid_h, script_w, script_h)
        cur_polys = eval_commands_to_polygons(cur_cmds)
        tracked_polys.append(cur_polys)
        tracked_clips.append(format_commands_as_clip_inner(cur_cmds))

        prev_gray = curr_gray
        prev_pts = detect_features(curr_gray, cur_polys)

    return tracked_polys, tracked_clips


def mask_to_ass_clip(
    mask_u8: np.ndarray,
    vid_w: int,
    vid_h: int,
    script_w: int,
    script_h: int,
    simplify_epsilon: float = 1.5,
    min_area: float = 16.0,
) -> str:
    """
    Convert an 8-bit binary mask (0/255) into a compact ASS `m x0 y0 l x1 y1 ...`
    vector clip string scaled to script resolution, preserving holes via RETR_CCOMP.
    """
    binary = ((mask_u8 > 127).astype(np.uint8)) * 255
    if cv2.countNonZero(binary) == 0:
        return ""

    contours, _ = cv2.findContours(binary, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return ""

    sx = float(script_w) / float(vid_w) if vid_w > 0 else 1.0
    sy = float(script_h) / float(vid_h) if vid_h > 0 else 1.0
    eps = max(0.0, float(simplify_epsilon))

    subpaths: List[str] = []
    for contour in contours:
        if abs(cv2.contourArea(contour)) < float(min_area):
            continue
        pts = contour.reshape(-1, 2).astype(np.float32)
        if sx != 1.0 or sy != 1.0:
            pts[:, 0] *= sx
            pts[:, 1] *= sy
        if eps > 0.0 and len(pts) > 3:
            approx = cv2.approxPolyDP(pts.reshape(-1, 1, 2), epsilon=eps, closed=True).reshape(-1, 2)
        else:
            approx = pts

        deduped: List[Tuple[int, int]] = []
        for px, py in approx:
            pt = (int(round(float(px))), int(round(float(py))))
            if not deduped or pt != deduped[-1]:
                deduped.append(pt)
        if len(deduped) > 1 and deduped[-1] == deduped[0]:
            deduped.pop()
        if len(deduped) < 3:
            continue

        cmds = [f"m {deduped[0][0]} {deduped[0][1]} l"]
        cmds.extend(f"{x} {y}" for x, y in deduped[1:])
        subpaths.append(" ".join(cmds))

    return " ".join(subpaths)


# Interactive 2-option checkbox selector & mode-specific widgets
_is_moving_init = OBJECT_MASK_MODE.startswith("Mask moving object")
cb_static = widgets.Checkbox(
    value=not _is_moving_init,
    description="Mask static object (existing \\clip workflow)",
    indent=False,
    layout=widgets.Layout(width="auto"),
)
cb_moving = widgets.Checkbox(
    value=_is_moving_init,
    description="Mask moving object (SAM 2 Video Predictor)",
    indent=False,
    layout=widgets.Layout(width="auto"),
)

w_start = widgets.IntText(value=START_FRAME, description="Start Frame:")
w_end = widgets.IntText(value=END_FRAME, description="End Frame (-1=end):")
w_offset = widgets.Text(value=START_TIME_OFFSET, description="Start Offset:")
w_dilate = widgets.IntSlider(value=MASK_DILATION_PX, min=0, max=20, description="Inpaint Dilate (px):")
w_pad = widgets.IntSlider(value=CROP_PADDING_PX, min=16, max=256, step=8, description="Crop Pad (px):")

# Mode-specific controls
w_clip = widgets.Textarea(value=ASS_CLIP_STRING, description="ASS \\clip:", layout=widgets.Layout(width="90%", height="60px"))
w_static_lk = widgets.Checkbox(
    value=bool(STATIC_RIGID_LK_TRACK),
    description="Enable rigid Lucas-Kanade optical-flow pan tracking for static \\clip",
    indent=False,
    layout=widgets.Layout(width="auto"),
)
static_box = widgets.VBox([w_clip, w_static_lk])

w_sam2_expand = widgets.IntSlider(value=SAM2_CLIP_EXPAND_PX, min=0, max=12, description="Clip Expand (px):")
w_sam2_eps = widgets.FloatSlider(value=SAM2_CLIP_EPSILON, min=0.5, max=8.0, step=0.5, description="Simplify Epsilon:")
w_sam2_area = widgets.FloatText(value=SAM2_MIN_CONTOUR_AREA, description="Min Area (px²):")
moving_hint = widgets.HTML(
    "<b>SAM 2 Mode:</b> After running this cell, run <b>Cell 3A</b> to click or draw a box around the moving object on Frame 0."
)
moving_box = widgets.VBox([moving_hint, widgets.HBox([w_sam2_expand, w_sam2_eps, w_sam2_area])])


def _refresh_mode_visibility():
    static_box.layout.display = "" if cb_static.value else "none"
    moving_box.layout.display = "" if cb_moving.value else "none"


_cb_lock = {"busy": False}


def _on_checkbox_change(change):
    global OBJECT_MASK_MODE
    if _cb_lock["busy"]:
        return
    _cb_lock["busy"] = True
    try:
        owner = change["owner"]
        new_val = bool(change["new"])
        if owner is cb_static:
            if new_val:
                cb_moving.value = False
            elif not cb_moving.value:
                cb_static.value = True
        elif owner is cb_moving:
            if new_val:
                cb_static.value = False
            elif not cb_static.value:
                cb_moving.value = True
        OBJECT_MASK_MODE = (
            "Mask moving object (SAM 2 Video Predictor)"
            if cb_moving.value
            else "Mask static object"
        )
        _refresh_mode_visibility()
    finally:
        _cb_lock["busy"] = False


def _sync_form_from_widgets(change=None):
    global ASS_CLIP_STRING, START_FRAME, END_FRAME, START_TIME_OFFSET
    global MASK_DILATION_PX, CROP_PADDING_PX, STATIC_RIGID_LK_TRACK
    global SAM2_CLIP_EXPAND_PX, SAM2_CLIP_EPSILON, SAM2_MIN_CONTOUR_AREA
    ASS_CLIP_STRING = w_clip.value
    START_FRAME = int(w_start.value)
    END_FRAME = int(w_end.value)
    START_TIME_OFFSET = w_offset.value
    MASK_DILATION_PX = int(w_dilate.value)
    CROP_PADDING_PX = int(w_pad.value)
    STATIC_RIGID_LK_TRACK = bool(w_static_lk.value)
    SAM2_CLIP_EXPAND_PX = int(w_sam2_expand.value)
    SAM2_CLIP_EPSILON = float(w_sam2_eps.value)
    SAM2_MIN_CONTOUR_AREA = float(w_sam2_area.value)


cb_static.observe(_on_checkbox_change, names="value")
cb_moving.observe(_on_checkbox_change, names="value")
for _w in (w_clip, w_start, w_end, w_offset, w_dilate, w_pad, w_static_lk, w_sam2_expand, w_sam2_eps, w_sam2_area):
    _w.observe(_sync_form_from_widgets, names="value")

_refresh_mode_visibility()
display(
    widgets.VBox([
        widgets.HTML("<b>Object Mask Mode (choose one):</b>"),
        widgets.HBox([cb_static, cb_moving]),
        widgets.HBox([w_start, w_end, w_offset]),
        widgets.HBox([w_dilate, w_pad]),
        static_box,
        moving_box,
    ])
)
print(f"Configuration loaded (Mode: {OBJECT_MASK_MODE}). Next, run Cell 3A.")

## 3. Object Masking & Preview

- **Cell 3A**: Loads the target frame sequence. If **Mask moving object (SAM 2 Video Predictor)** is checked, opens an interactive Frame 0 selector (`Positive Point`, `Negative Point`, `Bounding Box`, `Preview Mask`, `Reset Points`). If **Mask static object** is checked, skips interactive selection automatically.
- **Cell 3B**: Propagates the SAM 2 mask across all frames (or parses the static `\clip` polygon), generates per-frame ASS `\clip(...)` tags, computes the padded GPU crop bounding box, and previews First / Middle / Last frames.

In [ ]:
#@title 3A. Load Frames & Interactive SAM 2 Object Selection (Frame 0) { display-mode: "form" }
import contextlib
import os
import shutil
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

EFF_PLAY_RES_X = PLAY_RES_X if PLAY_RES_X > 0 else VIDEO_WIDTH
EFF_PLAY_RES_Y = PLAY_RES_Y if PLAY_RES_Y > 0 else VIDEO_HEIGHT
OFFSET_MS = parse_ass_time_to_ms(START_TIME_OFFSET)

eff_start_frame = max(0, min(TOTAL_FRAMES - 1, int(START_FRAME)))
eff_end_frame = (TOTAL_FRAMES - 1) if END_FRAME < 0 else max(eff_start_frame, min(TOTAL_FRAMES - 1, int(END_FRAME)))

FRAMES_BGR = []
cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, float(eff_start_frame))
for f_idx in range(eff_start_frame, eff_end_frame + 1):
    ok, frame = cap.read()
    if not ok or frame is None:
        raise RuntimeError(f"Failed to read video frame {f_idx}.")
    FRAMES_BGR.append(frame)
cap.release()

print(f"Loaded {len(FRAMES_BGR)} frames ({eff_start_frame} .. {eff_end_frame}).")

IS_MOVING_OBJECT_MODE = OBJECT_MASK_MODE.startswith("Mask moving object")

if not IS_MOVING_OBJECT_MODE:
    SAM2_SELECTION_STATE = None
    print("Mode: Mask static object -> skipping interactive SAM 2 selection. Run Cell 3B next.")
else:
    import torch
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches

    try:
        from IPython import get_ipython
        _ip = get_ipython()
        if _ip is not None:
            _ip.run_line_magic("matplotlib", "widget")
    except Exception:
        pass

    _content_root = globals().get("CONTENT_ROOT", "/content" if os.path.exists("/content") else os.getcwd())
    SAM2_FRAMES_DIR = os.path.join(_content_root, "sam2_frames")
    if os.path.exists(SAM2_FRAMES_DIR):
        shutil.rmtree(SAM2_FRAMES_DIR)
    os.makedirs(SAM2_FRAMES_DIR, exist_ok=True)

    for idx, frm in enumerate(FRAMES_BGR):
        cv2.imwrite(os.path.join(SAM2_FRAMES_DIR, f"{idx:05d}.jpg"), frm, [cv2.IMWRITE_JPEG_QUALITY, 95])

    _has_cuda = bool(globals().get("HAS_CUDA", torch.cuda.is_available()))
    _sam2_device = "cuda" if _has_cuda else "cpu"
    _dtype_name = globals().get("SAM2_AUTOCAST_DTYPE_NAME", "float16")
    _sam2_dtype = getattr(torch, _dtype_name, torch.float16) if _has_cuda else None
    _offload_cpu = bool(globals().get("OFFLOAD_VIDEO_TO_CPU", True))

    def _sam2_autocast_ctx():
        if _has_cuda and _sam2_dtype is not None:
            return torch.autocast(device_type="cuda", dtype=_sam2_dtype)
        return contextlib.nullcontext()

    if globals().get("SAM2_PREDICTOR") is None:
        from sam2.build_sam import build_sam2_video_predictor
        _ckpt = globals().get("SAM2_CKPT_PATH", os.path.join(_content_root, "checkpoints", "sam2.1_hiera_large.pt"))
        if not os.path.exists(_ckpt):
            raise FileNotFoundError(
                f"SAM 2.1 checkpoint missing at {_ckpt}. Enable SETUP_SAM2=True in Cell 1 and re-run Cell 1."
            )
        print(f"Loading SAM 2.1 Video Predictor on {_sam2_device}...")
        SAM2_PREDICTOR = build_sam2_video_predictor("configs/sam2.1/sam2.1_hiera_l.yaml", _ckpt, device=_sam2_device)

    with torch.inference_mode(), _sam2_autocast_ctx():
        SAM2_INFERENCE_STATE = SAM2_PREDICTOR.init_state(
            video_path=SAM2_FRAMES_DIR,
            offload_video_to_cpu=_offload_cpu,
        )

    first_rgb = cv2.cvtColor(FRAMES_BGR[0], cv2.COLOR_BGR2RGB)
    img_h, img_w = first_rgb.shape[:2]

    SAM2_SELECTION_STATE = {
        "points": [],
        "labels": [],
        "box_corners": [],
        "box": None,
        "preview_mask": None,
        "initialized": False,
    }

    mode_toggle = widgets.ToggleButtons(
        options=[
            ("Positive Point (1)", "pos"),
            ("Negative Point (0)", "neg"),
            ("Bounding Box (2 clicks)", "box"),
        ],
        value="pos",
        description="Tool:",
    )
    preview_btn = widgets.Button(description="Preview Mask", button_style="success", icon="eye")
    reset_btn = widgets.Button(description="Reset Points", button_style="warning", icon="trash")
    status_out = widgets.Output()

    plt.ioff()
    fig, ax = plt.subplots(figsize=(10, 6))
    plt.ion()
    try:
        fig.canvas.header_visible = False
    except Exception:
        pass

    def redraw_canvas():
        saved_xlim = ax.get_xlim() if SAM2_SELECTION_STATE["initialized"] else None
        saved_ylim = ax.get_ylim() if SAM2_SELECTION_STATE["initialized"] else None

        ax.clear()
        ax.imshow(first_rgb)

        if SAM2_SELECTION_STATE["preview_mask"] is not None:
            mask = SAM2_SELECTION_STATE["preview_mask"]
            overlay = np.zeros((img_h, img_w, 4), dtype=np.float32)
            overlay[mask] = [0.15, 0.75, 1.0, 0.45]
            ax.imshow(overlay)
            cnts, _ = cv2.findContours(mask.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in cnts:
                if len(cnt) >= 2:
                    pts_c = cnt.reshape(-1, 2)
                    ax.plot(pts_c[:, 0], pts_c[:, 1], color="#00FFFF", linewidth=1.5)

        pos_pts = [p for p, l in zip(SAM2_SELECTION_STATE["points"], SAM2_SELECTION_STATE["labels"]) if l == 1]
        neg_pts = [p for p, l in zip(SAM2_SELECTION_STATE["points"], SAM2_SELECTION_STATE["labels"]) if l == 0]
        if pos_pts:
            arr = np.array(pos_pts)
            ax.scatter(arr[:, 0], arr[:, 1], c="#00FF00", marker="*", s=220, edgecolors="black", linewidths=0.8, zorder=5)
        if neg_pts:
            arr = np.array(neg_pts)
            ax.scatter(arr[:, 0], arr[:, 1], c="#FF2222", marker="*", s=220, edgecolors="black", linewidths=0.8, zorder=5)

        if SAM2_SELECTION_STATE["box"] is not None:
            x1, y1, x2, y2 = SAM2_SELECTION_STATE["box"]
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="#00FFFF", linestyle="--", facecolor="none", zorder=4))
        elif len(SAM2_SELECTION_STATE["box_corners"]) == 1:
            cx, cy = SAM2_SELECTION_STATE["box_corners"][0]
            ax.scatter([cx], [cy], c="#00FFFF", marker="+", s=160, linewidths=2, zorder=5)

        ax.set_title(
            f"Frame {eff_start_frame} ({img_w}x{img_h}) — Pos: {len(pos_pts)} | Neg: {len(neg_pts)} | Box: {'Yes' if SAM2_SELECTION_STATE['box'] else 'No'}"
        )
        ax.axis("off")

        if saved_xlim is not None and saved_ylim is not None:
            ax.set_xlim(saved_xlim)
            ax.set_ylim(saved_ylim)
        SAM2_SELECTION_STATE["initialized"] = True
        fig.canvas.draw_idle()

    def on_canvas_click(event):
        if event.button != 1 or event.inaxes != ax or event.xdata is None or event.ydata is None:
            return
        toolbar = getattr(fig.canvas, "toolbar", None)
        if toolbar is not None and getattr(toolbar, "mode", ""):
            return
        x, y = float(event.xdata), float(event.ydata)
        mode = mode_toggle.value
        if mode == "pos":
            SAM2_SELECTION_STATE["points"].append([x, y])
            SAM2_SELECTION_STATE["labels"].append(1)
        elif mode == "neg":
            SAM2_SELECTION_STATE["points"].append([x, y])
            SAM2_SELECTION_STATE["labels"].append(0)
        elif mode == "box":
            SAM2_SELECTION_STATE["box_corners"].append([x, y])
            if len(SAM2_SELECTION_STATE["box_corners"]) == 2:
                c1, c2 = SAM2_SELECTION_STATE["box_corners"]
                SAM2_SELECTION_STATE["box"] = [min(c1[0], c2[0]), min(c1[1], c2[1]), max(c1[0], c2[0]), max(c1[1], c2[1])]
                SAM2_SELECTION_STATE["box_corners"] = []
        redraw_canvas()

    def on_reset_clicked(_):
        SAM2_SELECTION_STATE["points"].clear()
        SAM2_SELECTION_STATE["labels"].clear()
        SAM2_SELECTION_STATE["box_corners"].clear()
        SAM2_SELECTION_STATE["box"] = None
        SAM2_SELECTION_STATE["preview_mask"] = None
        SAM2_PREDICTOR.reset_state(SAM2_INFERENCE_STATE)
        with status_out:
            clear_output(wait=True)
            print("Selections cleared and SAM 2 inference state reset.")
        redraw_canvas()

    def on_preview_clicked(_):
        with status_out:
            clear_output(wait=True)
            if not SAM2_SELECTION_STATE["points"] and SAM2_SELECTION_STATE["box"] is None:
                print("Click on the image to add at least one point or bounding box first.")
                return
            SAM2_PREDICTOR.reset_state(SAM2_INFERENCE_STATE)
            pts_arr = np.array(SAM2_SELECTION_STATE["points"], dtype=np.float32) if SAM2_SELECTION_STATE["points"] else None
            lbl_arr = np.array(SAM2_SELECTION_STATE["labels"], dtype=np.int32) if SAM2_SELECTION_STATE["labels"] else None
            box_arr = np.array(SAM2_SELECTION_STATE["box"], dtype=np.float32) if SAM2_SELECTION_STATE["box"] is not None else None
            with torch.inference_mode(), _sam2_autocast_ctx():
                _, out_obj_ids, out_mask_logits = SAM2_PREDICTOR.add_new_points_or_box(
                    inference_state=SAM2_INFERENCE_STATE,
                    frame_idx=0,
                    obj_id=1,
                    points=pts_arr,
                    labels=lbl_arr,
                    box=box_arr,
                )
            mask_bool = (out_mask_logits[0].float() > 0.0).squeeze().cpu().numpy()
            SAM2_SELECTION_STATE["preview_mask"] = mask_bool
            fg_pct = 100.0 * float(mask_bool.sum()) / float(mask_bool.size)
            print(f"Preview mask ready for obj_id={int(out_obj_ids[0])} (foreground area: {fg_pct:.2f}%). Run Cell 3B to track all frames.")
        redraw_canvas()

    fig.canvas.mpl_connect("button_press_event", on_canvas_click)
    reset_btn.on_click(on_reset_clicked)
    preview_btn.on_click(on_preview_clicked)

    redraw_canvas()
    display(widgets.VBox([
        widgets.HBox([mode_toggle, preview_btn, reset_btn]),
        fig.canvas,
        status_out,
    ]))

In [ ]:
#@title 3B. Propagate Tracked Masks, Build Per-Frame \clip & Preview Coverage { display-mode: "form" }
import contextlib
import gc
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except Exception:
    pass

IS_MOVING_OBJECT_MODE = OBJECT_MASK_MODE.startswith("Mask moving object")
dil_ksize = 2 * max(0, int(MASK_DILATION_PX)) + 1
dil_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dil_ksize, dil_ksize)) if MASK_DILATION_PX > 0 else None

ORIG_MASKS = []
DILATED_MASKS = []
PER_FRAME_CLIPS = []

if IS_MOVING_OBJECT_MODE:
    import torch

    if not SAM2_SELECTION_STATE or (not SAM2_SELECTION_STATE["points"] and SAM2_SELECTION_STATE["box"] is None):
        raise ValueError("No points or box selected. Mark your moving object in Cell 3A first.")

    _has_cuda = bool(globals().get("HAS_CUDA", torch.cuda.is_available()))
    _dtype_name = globals().get("SAM2_AUTOCAST_DTYPE_NAME", "float16")
    _sam2_dtype = getattr(torch, _dtype_name, torch.float16) if _has_cuda else None
    _ctx = torch.autocast(device_type="cuda", dtype=_sam2_dtype) if (_has_cuda and _sam2_dtype is not None) else contextlib.nullcontext()

    SAM2_PREDICTOR.reset_state(SAM2_INFERENCE_STATE)
    pts_arr = np.array(SAM2_SELECTION_STATE["points"], dtype=np.float32) if SAM2_SELECTION_STATE["points"] else None
    lbl_arr = np.array(SAM2_SELECTION_STATE["labels"], dtype=np.int32) if SAM2_SELECTION_STATE["labels"] else None
    box_arr = np.array(SAM2_SELECTION_STATE["box"], dtype=np.float32) if SAM2_SELECTION_STATE["box"] is not None else None

    sam2_masks_by_idx = {}
    print(f"Propagating SAM 2.1 mask across {len(FRAMES_BGR)} frames...")
    with torch.inference_mode(), _ctx:
        SAM2_PREDICTOR.add_new_points_or_box(
            inference_state=SAM2_INFERENCE_STATE,
            frame_idx=0,
            obj_id=1,
            points=pts_arr,
            labels=lbl_arr,
            box=box_arr,
        )
        for frame_idx, _, out_mask_logits in SAM2_PREDICTOR.propagate_in_video(SAM2_INFERENCE_STATE):
            mask_bool = (out_mask_logits[0].float() > 0.0).squeeze().cpu().numpy()
            sam2_masks_by_idx[int(frame_idx)] = (mask_bool.astype(np.uint8)) * 255

    exp_px = max(0, int(SAM2_CLIP_EXPAND_PX))
    exp_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * exp_px + 1, 2 * exp_px + 1)) if exp_px > 0 else None

    for idx in range(len(FRAMES_BGR)):
        raw_m = sam2_masks_by_idx.get(idx)
        if raw_m is None or raw_m.shape[:2] != (VIDEO_HEIGHT, VIDEO_WIDTH):
            raw_m = np.zeros((VIDEO_HEIGHT, VIDEO_WIDTH), dtype=np.uint8) if raw_m is None else cv2.resize(raw_m, (VIDEO_WIDTH, VIDEO_HEIGHT), interpolation=cv2.INTER_NEAREST)

        m_orig = cv2.dilate(raw_m, exp_kernel) if (exp_kernel is not None and np.any(raw_m)) else raw_m
        clip_inner = mask_to_ass_clip(
            m_orig,
            VIDEO_WIDTH,
            VIDEO_HEIGHT,
            EFF_PLAY_RES_X,
            EFF_PLAY_RES_Y,
            simplify_epsilon=SAM2_CLIP_EPSILON,
            min_area=SAM2_MIN_CONTOUR_AREA,
        )
        # Re-rasterize simplified clip so ORIG_MASKS matches the exact ASS \clip boundary
        if clip_inner:
            _, simplified_polys, _ = parse_ass_clip_structured(clip_inner)
            m_orig_clip = rasterize_clip_mask(simplified_polys, VIDEO_WIDTH, VIDEO_HEIGHT, EFF_PLAY_RES_X, EFF_PLAY_RES_Y)
        else:
            m_orig_clip = np.zeros((VIDEO_HEIGHT, VIDEO_WIDTH), dtype=np.uint8)

        m_dil = cv2.dilate(m_orig_clip, dil_kernel) if (dil_kernel is not None and np.any(m_orig_clip)) else m_orig_clip.copy()
        ORIG_MASKS.append(m_orig_clip)
        DILATED_MASKS.append(m_dil)
        PER_FRAME_CLIPS.append(clip_inner)

    non_empty_clips = [c for c in PER_FRAME_CLIPS if c]
    if not non_empty_clips:
        raise ValueError("SAM 2 produced empty masks on all frames! Re-run Cell 3A with clearer points/box.")
    RAW_CLIP_INNER = non_empty_clips[0]

    # Release SAM 2 inference state so Cell 4 has full GPU VRAM
    try:
        SAM2_PREDICTOR.reset_state(SAM2_INFERENCE_STATE)
        del SAM2_INFERENCE_STATE
        if float(globals().get("VRAM_GB", 0.0)) <= 18.0:
            SAM2_PREDICTOR = None
        gc.collect()
        if _has_cuda:
            torch.cuda.empty_cache()
    except Exception:
        pass
else:
    BASE_CMDS, BASE_POLYS, RAW_CLIP_INNER = parse_ass_clip_structured(ASS_CLIP_STRING)
    if STATIC_RIGID_LK_TRACK:
        print(f"Tracking static \\clip polygon across {len(FRAMES_BGR)} frames using Lucas-Kanade optical flow...")
        PER_FRAME_POLYS, PER_FRAME_CLIPS = track_clip_contours_lk(
            FRAMES_BGR, BASE_CMDS, RAW_CLIP_INNER,
            VIDEO_WIDTH, VIDEO_HEIGHT, EFF_PLAY_RES_X, EFF_PLAY_RES_Y,
        )
    else:
        PER_FRAME_POLYS = [BASE_POLYS for _ in FRAMES_BGR]
        PER_FRAME_CLIPS = [RAW_CLIP_INNER for _ in FRAMES_BGR]

    for polys in PER_FRAME_POLYS:
        m_orig = rasterize_clip_mask(polys, VIDEO_WIDTH, VIDEO_HEIGHT, EFF_PLAY_RES_X, EFF_PLAY_RES_Y)
        m_dil = cv2.dilate(m_orig, dil_kernel) if dil_kernel is not None else m_orig.copy()
        ORIG_MASKS.append(m_orig)
        DILATED_MASKS.append(m_dil)

# Compute union GPU crop box across all active frames
union_ys, union_xs = [], []
for m_dil in DILATED_MASKS:
    ys, xs = np.nonzero(m_dil)
    if len(xs) > 0:
        union_xs.extend([int(xs.min()), int(xs.max()) + 1])
        union_ys.extend([int(ys.min()), int(ys.max()) + 1])

if not union_xs:
    raise ValueError("Mask is empty across all selected frames! Check your \\clip coordinates or SAM 2 selection.")

ux0, ux1 = min(union_xs), max(union_xs)
uy0, uy1 = min(union_ys), max(union_ys)

pad = max(16, int(CROP_PADDING_PX))
cx0 = max(0, ux0 - pad)
cy0 = max(0, uy0 - pad)
cx1 = min(VIDEO_WIDTH, ux1 + pad)
cy1 = min(VIDEO_HEIGHT, uy1 + pad)
if cx1 - cx0 < 64:
    cx0 = max(0, min(cx0, VIDEO_WIDTH - 64))
    cx1 = min(VIDEO_WIDTH, cx0 + 64)
if cy1 - cy0 < 64:
    cy0 = max(0, min(cy0, VIDEO_HEIGHT - 64))
    cy1 = min(VIDEO_HEIGHT, cy0 + 64)

CROP_BOX = (cx0, cy0, cx1, cy1)
crop_w, crop_h = cx1 - cx0, cy1 - cy0
active_frame_count = sum(1 for c in PER_FRAME_CLIPS if c)
print(f"Active Masks   : {active_frame_count}/{len(FRAMES_BGR)} frames")
print(f"GPU Crop Region: x={cx0}..{cx1}, y={cy0}..{cy1} ({crop_w}x{crop_h} px)")

# Preview First, Middle, and Last frames with mask overlay & GPU crop box
sample_indices = sorted(set([0, len(FRAMES_BGR) // 2, len(FRAMES_BGR) - 1]))
fig, axes = plt.subplots(1, len(sample_indices), figsize=(6 * len(sample_indices), 5))
if len(sample_indices) == 1:
    axes = [axes]

for ax_p, s_idx in zip(axes, sample_indices):
    f_rgb = cv2.cvtColor(FRAMES_BGR[s_idx], cv2.COLOR_BGR2RGB)
    overlay = f_rgb.copy().astype(np.float32)
    m_dil = DILATED_MASKS[s_idx] > 0
    if np.any(m_dil):
        overlay[m_dil] = overlay[m_dil] * 0.45 + np.array([255.0, 50.0, 50.0]) * 0.55
    overlay_u8 = np.clip(overlay, 0, 255).astype(np.uint8)
    cnts, _ = cv2.findContours(ORIG_MASKS[s_idx], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay_u8, cnts, -1, (0, 255, 100), 2)
    cv2.rectangle(overlay_u8, (cx0, cy0), (cx1 - 1, cy1 - 1), (255, 220, 0), 3)
    ax_p.imshow(overlay_u8)
    ax_p.set_title(f"Frame {eff_start_frame + s_idx}: Mask (Red) + Crop (Yellow)")
    ax_p.axis("off")

plt.tight_layout()
plt.show()

## 4. Temporally Consistent Video Inpainting (G4 / L4 / T4 GPU)

Selects and runs temporally consistent video inpainting strictly inside the padded crop box, automatically scaling temporal window sizes and precision to your GPU (`G4 96GB`, `L4 24GB`, or `T4 16GB`):
- **DiffuEraser + ProPainter Prior (G4 GPU Recommended)**: Uses ProPainter optical-flow initialization followed by Stable Diffusion 1.5 + BrushNet + Temporal Attention + PCM refinement for complex moving clouds/water/textures.
- **ProPainter + LaMa Spatial Refinement**: Combines ProPainter flow propagation with LaMa Fourier convolutions in deep mask centers plus temporal optical-flow smoothing.
- **ProPainter Full-Context (FP32 on G4 / FP16 on T4)**: Pure flow-guided transformer inpainting (great for crisp anime lines and clean gradients without diffusion hallucination).
- **Optical-Flow Temporal Fallback**: Bidirectional Farneback flow + Navier-Stokes if CUDA/weights are unavailable.

In [ ]:
#@title 4. Run Temporal Video Inpainting & Preview Comparison { display-mode: "form" }
import ast
import glob
import inspect
import math
import os
import subprocess
import sys
import tempfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

INPAINT_ENGINE = "Auto (Best for GPU: DiffuEraser on G4/L4, ProPainter+LaMa on T4)"  #@param ["Auto (Best for GPU: DiffuEraser on G4/L4, ProPainter+LaMa on T4)", "DiffuEraser + ProPainter Prior (G4 GPU Recommended - SOTA Diffusion)", "ProPainter Full-Context + LaMa Refinement (Sharp Hybrid)", "ProPainter AI (FP32 on G4 / FP16 on T4)", "Optical-Flow Temporal Fallback (Fast CPU/GPU)"]
SUBVIDEO_LENGTH = 0  #@param {type:"integer"}  # 0 = Auto (120 on G4, 80 on L4, 50 on T4)
NEIGHBOR_LENGTH = 0  #@param {type:"integer"}  # 0 = Auto (20 on G4, 15 on L4, 10 on T4)
REF_STRIDE = 0  #@param {type:"integer"}  # 0 = Auto (5 on G4, 8 on L4, 10 on T4)
PRECISION_MODE = "Auto (FP32 on G4, FP16 on L4/T4)"  #@param ["Auto (FP32 on G4, FP16 on L4/T4)", "FP32 (No Gradient Banding)", "FP16 (Faster / Lower VRAM)"]
DIFFUERASER_MAX_IMG_SIZE = 0  #@param {type:"integer"}  # 0 = Auto (1920 on G4, 1280 on L4, 960 on T4)
ENABLE_BOUNDARY_HARMONIZATION = True  #@param {type:"boolean"}

# Resolve hardware-aware defaults even if Cell 1 was skipped
_profile = globals().get("RECOMMENDED_PROFILE", {
    "subvideo_length": 50,
    "neighbor_length": 10,
    "ref_stride": 10,
    "use_fp16": True,
    "max_img_size": 960,
    "diffueraser_steps": 2,
})
EFF_SUBVIDEO_LENGTH = int(SUBVIDEO_LENGTH) if int(SUBVIDEO_LENGTH) > 0 else int(_profile["subvideo_length"])
EFF_NEIGHBOR_LENGTH = int(NEIGHBOR_LENGTH) if int(NEIGHBOR_LENGTH) > 0 else int(_profile["neighbor_length"])
EFF_REF_STRIDE = int(REF_STRIDE) if int(REF_STRIDE) > 0 else int(_profile["ref_stride"])
EFF_MAX_IMG_SIZE = int(DIFFUERASER_MAX_IMG_SIZE) if int(DIFFUERASER_MAX_IMG_SIZE) > 0 else int(_profile["max_img_size"])
if PRECISION_MODE.startswith("FP32"):
    EFF_USE_FP16 = False
elif PRECISION_MODE.startswith("FP16"):
    EFF_USE_FP16 = True
else:
    EFF_USE_FP16 = bool(_profile["use_fp16"])


def harmonize_boundary_colors(orig_frames_bgr, inpainted_frames_bgr, crop_masks):
    """
    Match brightness/tint of inpainted regions to untouched surrounding background
    using an outer perimeter ring, eliminating subtle VAE or FP16 seam shifts.
    """
    k3 = np.ones((3, 3), dtype=np.uint8)
    k11 = np.ones((11, 11), dtype=np.uint8)
    harmonized = []

    for orig, inp, msk in zip(orig_frames_bgr, inpainted_frames_bgr, crop_masks):
        m_bin = (msk > 0).astype(np.uint8)
        if not np.any(m_bin):
            harmonized.append(orig.copy())
            continue

        outer_ring = (cv2.dilate(m_bin, k11) > 0) & (cv2.dilate(m_bin, k3) == 0)
        inp_f = inp.astype(np.float32)
        orig_f = orig.astype(np.float32)

        if np.count_nonzero(outer_ring) >= 16:
            delta = np.median(orig_f[outer_ring] - inp_f[outer_ring], axis=0)
            delta = np.clip(delta, -12.0, 12.0)
            inp_f = np.clip(inp_f + delta[None, None, :], 0.0, 255.0)

        # Smooth 3px alpha blend across the dilated boundary into untouched original pixels
        soft_alpha = cv2.GaussianBlur(m_bin.astype(np.float32), (5, 5), 0)[..., None]
        soft_alpha = np.where(m_bin[..., None] > 0, np.maximum(soft_alpha, 0.85), soft_alpha * 0.5)
        blended = np.clip(inp_f * soft_alpha + orig_f * (1.0 - soft_alpha), 0.0, 255.0).astype(np.uint8)
        harmonized.append(blended)

    return harmonized


def _temporal_flow_blend_masked(orig_frames_bgr, cand_frames_bgr, crop_masks, alpha_self=0.6):
    """Enforce temporal smoothness inside masked regions using bidirectional Farneback flow."""
    n = len(cand_frames_bgr)
    if n <= 1:
        return [c.copy() for c in cand_frames_bgr]

    grays = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in cand_frames_bgr]
    h, w = grays[0].shape
    grid_x, grid_y = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    out_frames = []

    for i in range(n):
        m = crop_masks[i] > 0
        if not np.any(m):
            out_frames.append(orig_frames_bgr[i].copy())
            continue

        accum = cand_frames_bgr[i].astype(np.float32) * float(alpha_self)
        weight_sum = np.full((h, w, 1), float(alpha_self), dtype=np.float32)

        for offset in (-2, -1, 1, 2):
            j = i + offset
            if j < 0 or j >= n:
                continue
            flow = cv2.calcOpticalFlowFarneback(
                grays[i], grays[j], None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            warped = cv2.remap(
                cand_frames_bgr[j],
                grid_x + flow[:, :, 0],
                grid_y + flow[:, :, 1],
                interpolation=cv2.INTER_LINEAR,
                borderMode=cv2.BORDER_REPLICATE,
            )
            w_t = 0.40 if abs(offset) == 1 else 0.20
            accum += warped.astype(np.float32) * w_t
            weight_sum += w_t

        blended = np.clip(accum / weight_sum, 0, 255).astype(np.uint8)
        final_f = orig_frames_bgr[i].copy()
        final_f[m] = blended[m]
        out_frames.append(final_f)

    return out_frames


def inpaint_temporal_optical_flow(crop_frames_bgr, crop_masks):
    """
    Temporally consistent optical-flow guided video inpainting fallback:
    1. Spatial Navier-Stokes seed per frame.
    2. Bidirectional dense Farneback optical flow warping across +/-2 neighbors.
    """
    base_inpainted = [
        cv2.inpaint(crop_frames_bgr[i], crop_masks[i], inpaintRadius=5, flags=cv2.INPAINT_NS)
        for i in range(len(crop_frames_bgr))
    ]
    return _temporal_flow_blend_masked(crop_frames_bgr, base_inpainted, crop_masks, alpha_self=1.0)


def _pad_short_sequence(crop_frames_bgr, crop_masks, min_frames=8):
    """
    Pad ultra-short (< min_frames) frame/mask sequences by repeating the last frame
    so RAFT optical flow (which requires >= 2 frames / pairs) and temporal attention
    never fail on 1-frame or very short selections.
    """
    orig_n = len(crop_frames_bgr)
    if orig_n == 0:
        raise ValueError("Cannot run inpainting on an empty frame sequence.")
    if orig_n >= min_frames:
        return list(crop_frames_bgr), list(crop_masks), orig_n

    print(
        f"  [pad] Short sequence ({orig_n} frame(s)): padding to {min_frames} frames "
        f"for RAFT optical flow & temporal attention, then slicing back to {orig_n}."
    )
    proc_frames = [f.copy() for f in crop_frames_bgr]
    proc_masks = [m.copy() for m in crop_masks]
    last_f = crop_frames_bgr[-1]
    last_m = crop_masks[-1]
    while len(proc_frames) < min_frames:
        proc_frames.append(last_f.copy())
        proc_masks.append(last_m.copy())
    return proc_frames, proc_masks, orig_n


def patch_script_torchvision_io(script_path):
    """
    Inject an idempotent torchvision.io compatibility shim (read_video, write_video,
    read_video_timestamps) backed by OpenCV/imageio for modern torchvision (>= 0.25)
    where torchvision.io.read_video was removed.
    """
    if not script_path or not os.path.exists(script_path):
        return
    with open(script_path, "r", encoding="utf-8") as fh:
        src = fh.read()

    patch_marker = "# [ass-inpaint-mesh-torchvision-patch]"
    if patch_marker in src:
        return

    lines = src.splitlines(keepends=True)
    insert_idx = 0
    try:
        tree = ast.parse(src)
        for idx_stmt, stmt in enumerate(tree.body):
            is_docstring = (
                idx_stmt == 0
                and isinstance(stmt, ast.Expr)
                and isinstance(stmt.value, ast.Constant)
                and isinstance(stmt.value.value, str)
            )
            is_future = isinstance(stmt, ast.ImportFrom) and stmt.module == "__future__"
            if is_docstring or is_future:
                insert_idx = max(insert_idx, int(getattr(stmt, "end_lineno", stmt.lineno)))
            else:
                break
    except SyntaxError:
        for i, ln in enumerate(lines):
            stripped = ln.strip()
            if not stripped or stripped.startswith("#") or stripped.startswith("from __future__"):
                insert_idx = i + 1
            else:
                break

    if insert_idx == 0:
        while insert_idx < len(lines):
            stripped = lines[insert_idx].strip()
            if not stripped or stripped.startswith("#"):
                insert_idx += 1
            else:
                break

    tv_patch_block = """# [ass-inpaint-mesh-torchvision-patch]
try:
    import math as _ass_math
    import cv2 as _ass_cv2
    import numpy as _ass_np
    import torch as _ass_torch
    import torchvision.io as _tv_io

    if not hasattr(_tv_io, "read_video"):
        def _ass_read_video(filename, start_pts=0, end_pts=None, pts_unit="pts", output_format="THWC"):
            cap = _ass_cv2.VideoCapture(str(filename))
            if not cap.isOpened():
                raise RuntimeError(f"OpenCV VideoCapture failed to open video: {filename!r}")
            fps = float(cap.get(_ass_cv2.CAP_PROP_FPS) or 0.0)
            if not _ass_math.isfinite(fps) or fps <= 0.0:
                fps = 24.0
            unit = str(pts_unit or "pts").lower()
            frames = []
            idx = 0
            try:
                while True:
                    ok, bgr = cap.read()
                    if not ok or bgr is None:
                        break
                    if unit == "sec":
                        t = float(idx) / fps
                        if t >= float(start_pts) - 1e-6 and (end_pts is None or t <= float(end_pts) + 1e-6):
                            frames.append(_ass_cv2.cvtColor(bgr, _ass_cv2.COLOR_BGR2RGB))
                        if end_pts is not None and t > float(end_pts) + 1e-6:
                            break
                    else:
                        if idx >= int(start_pts) and (end_pts is None or idx <= int(end_pts)):
                            frames.append(_ass_cv2.cvtColor(bgr, _ass_cv2.COLOR_BGR2RGB))
                        if end_pts is not None and idx > int(end_pts):
                            break
                    idx += 1
            finally:
                cap.release()

            fmt = str(output_format or "THWC").upper()
            if frames:
                arr = _ass_np.ascontiguousarray(_ass_np.stack(frames, axis=0), dtype=_ass_np.uint8)
                vframes = _ass_torch.from_numpy(arr)
                if fmt == "TCHW":
                    vframes = vframes.permute(0, 3, 1, 2).contiguous()
            else:
                vframes = _ass_torch.empty((0, 3, 0, 0) if fmt == "TCHW" else (0, 0, 0, 3), dtype=_ass_torch.uint8)
            aframes = _ass_torch.empty((0, 0), dtype=_ass_torch.float32)
            return vframes, aframes, {"video_fps": float(fps)}

        _tv_io.read_video = _ass_read_video

    if not hasattr(_tv_io, "write_video"):
        def _ass_write_video(
            filename,
            video_array,
            fps,
            video_codec="libx264",
            options=None,
            audio_array=None,
            audio_fps=None,
            audio_codec=None,
            audio_options=None,
        ):
            if hasattr(video_array, "detach"):
                arr = video_array.detach().cpu().numpy()
            elif hasattr(video_array, "numpy"):
                arr = video_array.numpy()
            else:
                arr = _ass_np.asarray(video_array)
            arr = _ass_np.asarray(arr)
            if arr.dtype != _ass_np.uint8:
                arr = _ass_np.clip(arr, 0, 255).astype(_ass_np.uint8)
            if arr.ndim == 4 and arr.shape[-1] not in (1, 3, 4) and arr.shape[1] in (1, 3, 4):
                arr = _ass_np.transpose(arr, (0, 2, 3, 1))
            if arr.ndim == 4 and arr.shape[0] > 0:
                h, w = int(arr.shape[1]), int(arr.shape[2])
            else:
                raise ValueError(f"Expected non-empty 4D video_array, got shape {arr.shape}")

            fps_f = float(fps) if fps and float(fps) > 0.0 else 24.0
            try:
                import imageio as _ass_imageio
                with _ass_imageio.get_writer(
                    str(filename),
                    fps=fps_f,
                    codec=video_codec or "libx264",
                    macro_block_size=1,
                ) as writer:
                    for frm in arr:
                        writer.append_data(frm)
                return
            except Exception:
                pass

            fourcc = _ass_cv2.VideoWriter_fourcc(*"mp4v")
            writer = _ass_cv2.VideoWriter(str(filename), fourcc, fps_f, (w, h))
            if not writer.isOpened():
                raise RuntimeError(f"Failed to open VideoWriter for {filename!r}")
            try:
                for frm in arr:
                    if frm.ndim == 3 and frm.shape[-1] == 1:
                        bgr = _ass_cv2.cvtColor(frm[:, :, 0], _ass_cv2.COLOR_GRAY2BGR)
                    elif frm.ndim == 3 and frm.shape[-1] == 4:
                        bgr = _ass_cv2.cvtColor(frm, _ass_cv2.COLOR_RGBA2BGR)
                    else:
                        bgr = _ass_cv2.cvtColor(frm, _ass_cv2.COLOR_RGB2BGR)
                    writer.write(bgr)
            finally:
                writer.release()

        _tv_io.write_video = _ass_write_video

    if not hasattr(_tv_io, "read_video_timestamps"):
        def _ass_read_video_timestamps(filename, pts_unit="pts"):
            cap = _ass_cv2.VideoCapture(str(filename))
            if not cap.isOpened():
                raise RuntimeError(f"OpenCV VideoCapture failed to open video: {filename!r}")
            fps = float(cap.get(_ass_cv2.CAP_PROP_FPS) or 0.0)
            if not _ass_math.isfinite(fps) or fps <= 0.0:
                fps = 24.0
            count = 0
            try:
                while True:
                    ok, _ = cap.read()
                    if not ok:
                        break
                    count += 1
            finally:
                cap.release()
            unit = str(pts_unit or "pts").lower()
            pts = [float(i) / fps for i in range(count)] if unit == "sec" else list(range(count))
            return pts, float(fps)

        _tv_io.read_video_timestamps = _ass_read_video_timestamps
except Exception:
    pass
# [/ass-inpaint-mesh-torchvision-patch]
"""

    prefix = "".join(lines[:insert_idx])
    suffix = "".join(lines[insert_idx:])
    if prefix and not prefix.endswith("\n"):
        prefix += "\n"
    if suffix and not suffix.startswith("\n"):
        tv_patch_block += "\n"
    new_src = prefix + tv_patch_block + suffix
    with open(script_path, "w", encoding="utf-8") as fh:
        fh.write(new_src)
    print(f"  [patch] Added torchvision.io compatibility shim to {os.path.basename(script_path)}")


def run_propainter_on_crops(crop_frames_bgr, crop_masks):
    """Run sczhou/ProPainter on lossless cropped PNG frame & mask folders."""
    try:
        import torch
        cuda_ok = torch.cuda.is_available()
    except Exception:
        cuda_ok = False

    if not cuda_ok:
        raise RuntimeError("CUDA GPU unavailable for ProPainter.")
    prop_dir = globals().get("PROPAINTER_DIR", "/content/ProPainter")
    prop_script = os.path.join(prop_dir, "inference_propainter.py")
    if not os.path.exists(prop_script):
        raise FileNotFoundError(f"Missing ProPainter script: {prop_script}")

    patch_script_torchvision_io(prop_script)
    proc_frames, proc_masks, orig_n = _pad_short_sequence(crop_frames_bgr, crop_masks, min_frames=8)

    crop_h, crop_w = proc_frames[0].shape[:2]
    pad_right = (8 - (crop_w % 8)) % 8
    pad_bottom = (8 - (crop_h % 8)) % 8
    proc_w = crop_w + pad_right
    proc_h = crop_h + pad_bottom

    with tempfile.TemporaryDirectory() as work_dir:
        frames_dir = os.path.join(work_dir, "frames")
        masks_dir = os.path.join(work_dir, "masks")
        out_dir = os.path.join(work_dir, "results")
        os.makedirs(frames_dir, exist_ok=True)
        os.makedirs(masks_dir, exist_ok=True)
        os.makedirs(out_dir, exist_ok=True)

        for idx, (frm, msk) in enumerate(zip(proc_frames, proc_masks)):
            if pad_right > 0 or pad_bottom > 0:
                frm = cv2.copyMakeBorder(frm, 0, pad_bottom, 0, pad_right, cv2.BORDER_REPLICATE)
                msk = cv2.copyMakeBorder(msk, 0, pad_bottom, 0, pad_right, cv2.BORDER_CONSTANT, value=0)
            cv2.imwrite(os.path.join(frames_dir, f"{idx:05d}.png"), frm)
            cv2.imwrite(os.path.join(masks_dir, f"{idx:05d}.png"), msk)

        cmd = [
            sys.executable,
            prop_script,
            "--video", frames_dir,
            "--mask", masks_dir,
            "--output", out_dir,
            "--width", str(proc_w),
            "--height", str(proc_h),
            "--subvideo_length", str(max(10, EFF_SUBVIDEO_LENGTH)),
            "--neighbor_length", str(max(4, min(EFF_NEIGHBOR_LENGTH, len(proc_frames)))),
            "--ref_stride", str(max(4, min(EFF_REF_STRIDE, len(proc_frames)))),
            "--save_frames",
        ]
        if EFF_USE_FP16:
            cmd.append("--fp16")

        print("Running ProPainter:", " ".join(cmd))
        proc = subprocess.run(cmd, cwd=prop_dir, capture_output=True, text=True, check=False)
        if proc.returncode != 0:
            raise RuntimeError(f"ProPainter exited with code {proc.returncode}:\n{proc.stderr[-1500:]}")

        png_files = sorted(glob.glob(os.path.join(out_dir, "**", "frames", "*.png"), recursive=True))
        if len(png_files) >= orig_n:
            results = []
            for p in png_files[:orig_n]:
                img = cv2.imread(p, cv2.IMREAD_COLOR)
                if img.shape[:2] != (proc_h, proc_w):
                    img = cv2.resize(img, (proc_w, proc_h), interpolation=cv2.INTER_LINEAR)
                results.append(img[:crop_h, :crop_w].copy())
            return results

        mp4_files = sorted(glob.glob(os.path.join(out_dir, "**", "inpaint_out.mp4"), recursive=True))
        if mp4_files:
            results = []
            cap_out = cv2.VideoCapture(mp4_files[0])
            while len(results) < orig_n:
                ok, img = cap_out.read()
                if not ok or img is None:
                    break
                if img.shape[:2] != (proc_h, proc_w):
                    img = cv2.resize(img, (proc_w, proc_h), interpolation=cv2.INTER_LINEAR)
                results.append(img[:crop_h, :crop_w].copy())
            cap_out.release()
            if len(results) == orig_n:
                return results

        raise RuntimeError("ProPainter completed but output frames were not found.")


def run_propainter_plus_lama(crop_frames_bgr, crop_masks):
    """
    Hybrid engine: ProPainter flow propagation + SimpleLama GPU Fourier refinement
    deep inside large masks + bidirectional optical-flow temporal smoothing.
    """
    prop_results = run_propainter_on_crops(crop_frames_bgr, crop_masks)
    try:
        import torch
        from simple_lama_inpainting import SimpleLama

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        sig = inspect.signature(SimpleLama.__init__)
        lama = SimpleLama(device=device) if "device" in sig.parameters else SimpleLama()
    except Exception as exc:
        print(f"SimpleLama unavailable ({exc}); returning pure ProPainter output.")
        return prop_results

    crop_h, crop_w = crop_frames_bgr[0].shape[:2]
    hybrid_frames = []
    for orig, prop_f, msk in zip(crop_frames_bgr, prop_results, crop_masks):
        m_u8 = (msk > 0).astype(np.uint8) * 255
        if not np.any(m_u8):
            hybrid_frames.append(orig.copy())
            continue

        pil_img = Image.fromarray(cv2.cvtColor(orig, cv2.COLOR_BGR2RGB))
        pil_msk = Image.fromarray(m_u8, mode="L")
        lama_pil = lama(pil_img, pil_msk)
        if lama_pil.size != (crop_w, crop_h):
            lama_pil = lama_pil.crop((0, 0, crop_w, crop_h))
        lama_bgr = cv2.cvtColor(np.asarray(lama_pil, dtype=np.uint8), cv2.COLOR_RGB2BGR)

        # Trust ProPainter along mask boundary; blend LaMa sharpness toward deep interior
        dist = cv2.distanceTransform((msk > 0).astype(np.uint8), cv2.DIST_L2, 5)
        w_lama = np.clip((dist - 6.0) / 18.0, 0.0, 0.55)[..., None]
        blended = np.clip(
            lama_bgr.astype(np.float32) * w_lama + prop_f.astype(np.float32) * (1.0 - w_lama),
            0.0,
            255.0,
        ).astype(np.uint8)
        hybrid_frames.append(blended)

    return _temporal_flow_blend_masked(crop_frames_bgr, hybrid_frames, crop_masks, alpha_self=0.65)


def patch_diffueraser_for_modern_diffusers(de_dir):
    """
    Patch vendored DiffuEraser UNetMotionModel / UNet2DConditionModel for modern
    diffusers (>=0.31, Python 3.12/3.13) where load_lora_weights calls unet.load_lora_adapter,
    patch diffueraser/diffueraser.py to respect ASS_DIFFUERASER_DTYPE (FP32/BF16/FP16),
    and inject the torchvision.io compatibility shim into run_diffueraser.py.
    """
    import re as _re

    patch_marker_v1 = "# [ass-inpaint-mesh-patch]"
    patch_marker = "# [ass-inpaint-mesh-patch-v2]"
    import_snippet = (
        "try:\n"
        "    from diffusers.loaders.peft import PeftAdapterMixin\n"
        "except ImportError:\n"
        "    from diffusers.loaders import PeftAdapterMixin\n"
    )

    for rel_path, cls_name in (
        (os.path.join("libs", "unet_motion_model.py"), "UNetMotionModel"),
        (os.path.join("libs", "unet_2d_condition.py"), "UNet2DConditionModel"),
    ):
        fpath = os.path.join(de_dir, rel_path)
        if not os.path.exists(fpath):
            continue
        with open(fpath, "r", encoding="utf-8") as fh:
            src = fh.read()
        orig_src = src

        if "PeftAdapterMixin" not in src:
            cls_match = _re.search(rf"^class\s+{cls_name}\s*\(", src, flags=_re.MULTILINE)
            if cls_match:
                src = src[: cls_match.start()] + import_snippet + "\n" + src[cls_match.start() :]
            else:
                src = import_snippet + "\n" + src

        def _add_mixin(m):
            bases = [b.strip() for b in m.group(1).split(",") if b.strip()]
            if "PeftAdapterMixin" not in bases:
                if "UNet2DConditionLoadersMixin" in bases:
                    idx = bases.index("UNet2DConditionLoadersMixin") + 1
                    bases.insert(idx, "PeftAdapterMixin")
                else:
                    bases.append("PeftAdapterMixin")
            return f"class {cls_name}(" + ", ".join(bases) + "):"

        src = _re.sub(rf"class\s+{cls_name}\s*\(([^)]*)\)\s*:", _add_mixin, src, count=1)

        if cls_name == "UNetMotionModel" and patch_marker not in src:
            if patch_marker_v1 in src:
                src = src[: src.index(patch_marker_v1)].rstrip()
            compat_block = "\n\n" + patch_marker + """
try:
    import diffusers.loaders.peft as _dl_peft
    if hasattr(_dl_peft, "_SET_ADAPTER_SCALE_FN_MAPPING") and "UNetMotionModel" not in _dl_peft._SET_ADAPTER_SCALE_FN_MAPPING:
        if hasattr(_dl_peft, "_maybe_expand_lora_scales"):
            _dl_peft._SET_ADAPTER_SCALE_FN_MAPPING["UNetMotionModel"] = _dl_peft._maybe_expand_lora_scales
except Exception:
    pass


def _ass_is_lora_weight_dict(obj):
    if not isinstance(obj, dict) or not obj:
        return False
    sufs = (
        ".lora_linear_layer.down.weight",
        ".lora.down.weight",
        "_lora.down.weight",
        ".lora_A.weight",
        ".lora_down.weight",
        ".down.weight",
    )
    return any(isinstance(k, str) and k.endswith(sufs) for k in obj.keys())


def _ass_direct_merge_lora(model, lora_src, **kwargs):
    import os as _os
    import torch as _torch

    if isinstance(lora_src, dict):
        sd = lora_src
    elif isinstance(lora_src, str) and _os.path.isfile(lora_src):
        if lora_src.endswith(".safetensors"):
            from safetensors.torch import load_file as _st_load
            sd = _st_load(lora_src)
        else:
            sd = _torch.load(lora_src, map_location="cpu")
    else:
        raise ValueError(f"Unsupported LoRA source for direct merge: {type(lora_src)}")

    prefix = kwargs.get("prefix")
    if prefix and any(isinstance(k, str) and k.startswith(f"{prefix}.") for k in sd):
        sd = {k: v for k, v in sd.items() if isinstance(k, str) and k.startswith(f"{prefix}.")}

    mods = {name: mod for name, mod in model.named_modules() if hasattr(mod, "weight")}
    mod_by_flat = {name.replace(".", "_"): name for name in mods}

    def _norm_base(key):
        k = key[len("unet.") :] if key.startswith("unet.") else key
        if k.endswith(".alpha"):
            k = k[: -len(".alpha")]
        for suf in (
            ".lora_linear_layer.down.weight",
            ".lora_linear_layer.up.weight",
            ".lora.down.weight",
            ".lora.up.weight",
            "_lora.down.weight",
            "_lora.up.weight",
            ".lora_A.weight",
            ".lora_B.weight",
            ".lora_down.weight",
            ".lora_up.weight",
            ".down.weight",
            ".up.weight",
        ):
            if k.endswith(suf):
                k = k[: -len(suf)]
                break
        k = k.replace(".processor.", ".")
        if k.endswith(".to_out"):
            k = k + ".0"
        if k.startswith("lora_unet_"):
            cand = k[len("lora_unet_") :]
            if cand in mod_by_flat:
                return mod_by_flat[cand]
        if k in mods:
            return k
        k_flat = k.replace(".", "_")
        if k_flat in mod_by_flat:
            return mod_by_flat[k_flat]
        return k

    net_alphas = kwargs.get("network_alphas") or {}
    pairs = {}
    alphas = {}
    down_sufs = (
        ".lora_linear_layer.down.weight",
        ".lora.down.weight",
        "_lora.down.weight",
        ".lora_A.weight",
        ".lora_down.weight",
        ".down.weight",
    )
    up_sufs = (
        ".lora_linear_layer.up.weight",
        ".lora.up.weight",
        "_lora.up.weight",
        ".lora_B.weight",
        ".lora_up.weight",
        ".up.weight",
    )
    for k, v in sd.items():
        if not isinstance(k, str):
            continue
        if k.endswith(".alpha"):
            alphas[_norm_base(k)] = float(v.item() if hasattr(v, "item") else v)
            continue
        if k.endswith(down_sufs):
            pairs.setdefault(_norm_base(k), {})["down"] = v
        elif k.endswith(up_sufs):
            pairs.setdefault(_norm_base(k), {})["up"] = v

    for ak, av in net_alphas.items():
        if isinstance(ak, str):
            alphas[_norm_base(ak)] = float(av.item() if hasattr(av, "item") else av)

    merged = 0
    alpha_hits = 0
    sample_scale = None
    with _torch.no_grad():
        for base_k, wdict in pairs.items():
            if "down" not in wdict or "up" not in wdict:
                continue
            target_mod = mods.get(base_k)
            if target_mod is None or not hasattr(target_mod, "weight"):
                continue
            down_w = wdict["down"].to(device=target_mod.weight.device, dtype=_torch.float32)
            up_w = wdict["up"].to(device=target_mod.weight.device, dtype=_torch.float32)
            rank = max(1, int(down_w.shape[0]))
            if base_k in alphas:
                alpha = alphas[base_k]
                alpha_hits += 1
            else:
                alpha = float(rank)
            scale = alpha / float(rank)
            if sample_scale is None:
                sample_scale = scale
            if target_mod.weight.ndim == 2:
                delta = (up_w @ down_w) * scale
            else:
                delta = (up_w.flatten(1) @ down_w.flatten(1)).view_as(target_mod.weight) * scale
            target_mod.weight.data.add_(delta.to(dtype=target_mod.weight.dtype))
            merged += 1

    model._ass_direct_lora_merged = merged > 0
    scale_str = f"{sample_scale:.4f}" if sample_scale is not None else "n/a"
    print(
        f"  [patch] Directly merged {merged}/{len(pairs)} LoRA layers into {type(model).__name__} "
        f"(alphas matched={alpha_hits}/{len(pairs)}, sample scale={scale_str})"
    )
    if len(pairs) > 0 and merged < len(pairs):
        missing_k = [k for k in pairs if k not in mods][:5]
        raise RuntimeError(
            f"Incomplete LoRA merge into {type(model).__name__}: {merged}/{len(pairs)} merged; "
            f"unmatched sample={missing_k}"
        )


_orig_unet_load_lora = getattr(UNetMotionModel, "load_lora_adapter", None)
_orig_unet_fuse_lora = getattr(UNetMotionModel, "fuse_lora", None)
_orig_unet_set_adapters = getattr(UNetMotionModel, "set_adapters", None)


def _compat_load_lora_adapter(self, pretrained_model_name_or_path_or_dict, *args, **kwargs):
    import os as _os
    if _ass_is_lora_weight_dict(pretrained_model_name_or_path_or_dict) or (
        isinstance(pretrained_model_name_or_path_or_dict, str) and _os.path.isfile(pretrained_model_name_or_path_or_dict)
    ):
        _ass_direct_merge_lora(self, pretrained_model_name_or_path_or_dict, **kwargs)
        return
    if _orig_unet_load_lora is not None:
        return _orig_unet_load_lora(self, pretrained_model_name_or_path_or_dict, *args, **kwargs)
    _ass_direct_merge_lora(self, pretrained_model_name_or_path_or_dict, **kwargs)


def _compat_fuse_lora(self, *args, **kwargs):
    if getattr(self, "_ass_direct_lora_merged", False):
        return
    if _orig_unet_fuse_lora is not None:
        return _orig_unet_fuse_lora(self, *args, **kwargs)


def _compat_set_adapters(self, *args, **kwargs):
    if getattr(self, "_ass_direct_lora_merged", False):
        return
    if _orig_unet_set_adapters is not None:
        return _orig_unet_set_adapters(self, *args, **kwargs)


UNetMotionModel.load_lora_adapter = _compat_load_lora_adapter
UNetMotionModel.fuse_lora = _compat_fuse_lora
UNetMotionModel.set_adapters = _compat_set_adapters
"""
            src = src.rstrip() + compat_block

        if src != orig_src:
            with open(fpath, "w", encoding="utf-8") as fh:
                fh.write(src)
            print(f"  [patch] Updated {rel_path} for modern diffusers PeftAdapterMixin compatibility (v2)")

    de_core_path = os.path.join(de_dir, "diffueraser", "diffueraser.py")
    dtype_marker = "# [ass-inpaint-mesh-dtype-patch]"
    if os.path.exists(de_core_path):
        with open(de_core_path, "r", encoding="utf-8") as fh:
            de_src = fh.read()
        if dtype_marker not in de_src and "torch.float16" in de_src:
            de_new = de_src.replace("torch.float16", "_ASS_WEIGHT_DTYPE")
            dtype_header = (
                f"{dtype_marker}\n"
                "_ASS_DTYPE_NAME = __import__('os').environ.get('ASS_DIFFUERASER_DTYPE', 'float16').strip().lower()\n"
                "_ASS_WEIGHT_DTYPE = getattr(torch, _ASS_DTYPE_NAME, torch.float16)\n"
            )
            de_new, n_ins = _re.subn(
                r"^(import\s+torch\b[^\n]*\n)",
                lambda m: m.group(1) + dtype_header,
                de_new,
                count=1,
                flags=_re.MULTILINE,
            )
            if n_ins == 0:
                de_new = "import torch\n" + dtype_header + de_new
            with open(de_core_path, "w", encoding="utf-8") as fh:
                fh.write(de_new)
            print("  [patch] Updated diffueraser/diffueraser.py to honor ASS_DIFFUERASER_DTYPE (FP32/BF16/FP16)")

    patch_script_torchvision_io(os.path.join(de_dir, "run_diffueraser.py"))


def run_diffueraser_on_crops(crop_frames_bgr, crop_masks):
    """
    Run lixiaowen-xw/DiffuEraser (ProPainter flow prior + SD1.5 BrushNet +
    AnimateDiff Temporal Attention + PCM) on cropped video & mask sequences.
    """
    try:
        import torch
        cuda_ok = torch.cuda.is_available()
    except Exception:
        cuda_ok = False

    if not cuda_ok:
        raise RuntimeError("CUDA GPU unavailable for DiffuEraser.")

    de_dir = globals().get("DIFFUERASER_DIR", "/content/DiffuEraser")
    de_script = os.path.join(de_dir, "run_diffueraser.py")
    if not os.path.exists(de_script):
        raise FileNotFoundError(f"Missing DiffuEraser script: {de_script}")

    patch_diffueraser_for_modern_diffusers(de_dir)
    proc_frames, proc_masks, orig_n = _pad_short_sequence(crop_frames_bgr, crop_masks, min_frames=22)

    weights_root = os.path.join(de_dir, "weights")
    crop_h, crop_w = proc_frames[0].shape[:2]
    pad_right = (8 - (crop_w % 8)) % 8
    pad_bottom = (8 - (crop_h % 8)) % 8
    proc_w = crop_w + pad_right
    proc_h = crop_h + pad_bottom
    fps_val = float(globals().get("VIDEO_FPS", 24.0) or 24.0)
    temp_fps = max(1, int(round(fps_val)))

    if not EFF_USE_FP16:
        de_dtype = "float32"
    elif getattr(torch.cuda, "is_bf16_supported", lambda: False)() and float(globals().get("VRAM_GB", 0.0)) >= 22.0:
        de_dtype = "bfloat16"
    else:
        de_dtype = "float16"
    run_env = os.environ.copy()
    run_env["ASS_DIFFUERASER_DTYPE"] = de_dtype
    print(f"  [diffueraser] Precision: {de_dtype}")

    # Probe supported flags in the cloned run_diffueraser.py
    help_proc = subprocess.run([sys.executable, de_script, "--help"], cwd=de_dir, env=run_env, capture_output=True, text=True, check=False)
    help_txt = (help_proc.stdout or "") + "\n" + (help_proc.stderr or "")

    with tempfile.TemporaryDirectory() as work_dir:
        vid_in_path = os.path.join(work_dir, "crop_input.mp4")
        msk_in_path = os.path.join(work_dir, "crop_mask.mp4")
        save_dir = os.path.join(work_dir, "results")
        os.makedirs(save_dir, exist_ok=True)

        # Write high-quality mp4 inputs for run_diffueraser.py
        try:
            import imageio
            with imageio.get_writer(vid_in_path, fps=temp_fps, codec="libx264", quality=10, macro_block_size=1) as w_v, \
                 imageio.get_writer(msk_in_path, fps=temp_fps, codec="libx264", quality=10, macro_block_size=1) as w_m:
                for frm, msk in zip(proc_frames, proc_masks):
                    if pad_right > 0 or pad_bottom > 0:
                        frm = cv2.copyMakeBorder(frm, 0, pad_bottom, 0, pad_right, cv2.BORDER_REPLICATE)
                        msk = cv2.copyMakeBorder(msk, 0, pad_bottom, 0, pad_right, cv2.BORDER_CONSTANT, value=0)
                    w_v.append_data(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
                    w_m.append_data(cv2.cvtColor(msk, cv2.COLOR_GRAY2RGB))
        except Exception:
            fourcc = cv2.VideoWriter_fourcc(*"mp4v")
            wv = cv2.VideoWriter(vid_in_path, fourcc, float(temp_fps), (proc_w, proc_h))
            wm = cv2.VideoWriter(msk_in_path, fourcc, float(temp_fps), (proc_w, proc_h))
            for frm, msk in zip(proc_frames, proc_masks):
                if pad_right > 0 or pad_bottom > 0:
                    frm = cv2.copyMakeBorder(frm, 0, pad_bottom, 0, pad_right, cv2.BORDER_REPLICATE)
                    msk = cv2.copyMakeBorder(msk, 0, pad_bottom, 0, pad_right, cv2.BORDER_CONSTANT, value=0)
                wv.write(frm)
                wm.write(cv2.cvtColor(msk, cv2.COLOR_GRAY2BGR))
            wv.release()
            wm.release()

        vid_len_arg = max(22, len(proc_frames), int(math.ceil(len(proc_frames) / float(temp_fps))) + 2)
        cmd = [
            sys.executable,
            de_script,
            "--input_video", vid_in_path,
            "--input_mask", msk_in_path,
            "--save_path", save_dir,
            "--video_length", str(vid_len_arg),
            "--mask_dilation_iter", "1",  # Cancels DiffuEraser read_mask's unconditional 1px cv2.erode
            "--max_img_size", str(min(EFF_MAX_IMG_SIZE, max(384, proc_w, proc_h))),
            "--base_model_path", os.path.join(weights_root, "stable-diffusion-v1-5"),
            "--vae_path", os.path.join(weights_root, "sd-vae-ft-mse"),
            "--diffueraser_path", os.path.join(weights_root, "diffuEraser"),
            "--propainter_model_dir", os.path.join(weights_root, "propainter"),
        ]
        if "--pcm_weights_path" in help_txt:
            cmd.extend(["--pcm_weights_path", os.path.join(weights_root, "PCM_Weights")])
        if "--ref_stride" in help_txt:
            cmd.extend(["--ref_stride", str(max(4, min(EFF_REF_STRIDE, len(proc_frames))))])
        if "--neighbor_length" in help_txt:
            cmd.extend(["--neighbor_length", str(max(4, min(EFF_NEIGHBOR_LENGTH, len(proc_frames))))])
        if "--subvideo_length" in help_txt:
            cmd.extend(["--subvideo_length", str(max(16, EFF_SUBVIDEO_LENGTH))])

        print("Running DiffuEraser:", " ".join(cmd))
        proc = subprocess.run(cmd, cwd=de_dir, env=run_env, capture_output=True, text=True, check=False)
        for _line in ((proc.stdout or "") + "\n" + (proc.stderr or "")).splitlines():
            if "[patch]" in _line:
                print(_line)
        if proc.returncode != 0:
            raise RuntimeError(f"DiffuEraser exited with code {proc.returncode}:\n{proc.stderr[-1500:]}")

        # Prefer final diffueraser result over intermediate priori.mp4
        mp4_candidates = sorted(glob.glob(os.path.join(save_dir, "**", "*.mp4"), recursive=True))
        result_mp4s = [p for p in mp4_candidates if "priori" not in os.path.basename(p).lower()]
        target_mp4 = result_mp4s[0] if result_mp4s else (mp4_candidates[0] if mp4_candidates else None)
        if not target_mp4:
            raise RuntimeError("DiffuEraser finished but no output .mp4 was found in save_path.")

        results = []
        cap_out = cv2.VideoCapture(target_mp4)
        while len(results) < orig_n:
            ok, img = cap_out.read()
            if not ok or img is None:
                break
            if img.shape[:2] != (proc_h, proc_w):
                img = cv2.resize(img, (proc_w, proc_h), interpolation=cv2.INTER_CUBIC)
            results.append(img[:crop_h, :crop_w].copy())
        cap_out.release()

        if len(results) != orig_n:
            raise RuntimeError(f"Expected {orig_n} DiffuEraser frames, decoded {len(results)}.")
        return results


cx0, cy0, cx1, cy1 = CROP_BOX
CROP_FRAMES_BGR = [f[cy0:cy1, cx0:cx1].copy() for f in FRAMES_BGR]
CROP_MASKS = [m[cy0:cy1, cx0:cx1].copy() for m in DILATED_MASKS]

# Resolve Auto engine selection based on detected VRAM
if INPAINT_ENGINE.startswith("Auto"):
    vram_detected = float(globals().get("VRAM_GB", 0.0))
    if vram_detected >= 22.0 and os.path.exists(globals().get("DIFFUERASER_DIR", "/content/DiffuEraser")):
        _target_mode = "diffueraser"
    elif globals().get("HAS_CUDA", False):
        _target_mode = "propainter_lama"
    else:
        _target_mode = "optical_flow"
elif INPAINT_ENGINE.startswith("DiffuEraser"):
    _target_mode = "diffueraser"
elif "LaMa" in INPAINT_ENGINE:
    _target_mode = "propainter_lama"
elif INPAINT_ENGINE.startswith("ProPainter"):
    _target_mode = "propainter"
else:
    _target_mode = "optical_flow"

precision_label = "FP16" if EFF_USE_FP16 else "FP32"

if _target_mode == "diffueraser":
    try:
        INPAINTED_CROPS = run_diffueraser_on_crops(CROP_FRAMES_BGR, CROP_MASKS)
        USED_INPAINT_ENGINE = f"DiffuEraser + ProPainter Prior ({globals().get('GPU_TIER', 'GPU')})"
    except Exception as exc:
        print(f"DiffuEraser skipped/failed ({exc}). Falling back to ProPainter + LaMa...")
        try:
            INPAINTED_CROPS = run_propainter_plus_lama(CROP_FRAMES_BGR, CROP_MASKS)
            USED_INPAINT_ENGINE = f"ProPainter + LaMa Fallback ({precision_label})"
        except Exception as exc2:
            print(f"ProPainter skipped/failed ({exc2}). Falling back to Optical-Flow Temporal Inpainting...")
            INPAINTED_CROPS = inpaint_temporal_optical_flow(CROP_FRAMES_BGR, CROP_MASKS)
            USED_INPAINT_ENGINE = "Optical-Flow Temporal Fallback"
elif _target_mode == "propainter_lama":
    try:
        INPAINTED_CROPS = run_propainter_plus_lama(CROP_FRAMES_BGR, CROP_MASKS)
        USED_INPAINT_ENGINE = f"ProPainter + LaMa Hybrid ({precision_label})"
    except Exception as exc:
        print(f"ProPainter + LaMa skipped/failed ({exc}). Falling back to Optical-Flow Temporal Inpainting...")
        INPAINTED_CROPS = inpaint_temporal_optical_flow(CROP_FRAMES_BGR, CROP_MASKS)
        USED_INPAINT_ENGINE = "Optical-Flow Temporal Fallback"
elif _target_mode == "propainter":
    try:
        INPAINTED_CROPS = run_propainter_on_crops(CROP_FRAMES_BGR, CROP_MASKS)
        USED_INPAINT_ENGINE = f"ProPainter Full-Context ({precision_label})"
    except Exception as exc:
        print(f"ProPainter skipped/failed ({exc}). Falling back to Optical-Flow Temporal Inpainting...")
        INPAINTED_CROPS = inpaint_temporal_optical_flow(CROP_FRAMES_BGR, CROP_MASKS)
        USED_INPAINT_ENGINE = "Optical-Flow Temporal Fallback"
else:
    INPAINTED_CROPS = inpaint_temporal_optical_flow(CROP_FRAMES_BGR, CROP_MASKS)
    USED_INPAINT_ENGINE = "Optical-Flow Temporal"

if ENABLE_BOUNDARY_HARMONIZATION:
    INPAINTED_CROPS = harmonize_boundary_colors(CROP_FRAMES_BGR, INPAINTED_CROPS, CROP_MASKS)

print(f"Inpainting finished ({USED_INPAINT_ENGINE}) for {len(INPAINTED_CROPS)} frames.")

# Display side-by-side comparison on the middle frame
sample_idx = len(INPAINTED_CROPS) // 2
orig_sample_rgb = cv2.cvtColor(CROP_FRAMES_BGR[sample_idx], cv2.COLOR_BGR2RGB)
inp_sample_rgb = cv2.cvtColor(INPAINTED_CROPS[sample_idx], cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(orig_sample_rgb)
axes[0].set_title(f"Original Crop (Frame {eff_start_frame + sample_idx})")
axes[0].axis("off")

axes[1].imshow(inp_sample_rgb)
axes[1].set_title(f"Inpainted via {USED_INPAINT_ENGINE}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5. Frame-by-Frame ASS Vector Mesh Generator (Anti-Lag Optimization)

Samples the inpainted pixels inside each processed frame's `\clip(...)` region, applies **Greedy 2D Rectangle Merging** (`m x1 y1 l x2 y1 x2 y2 x1 y2`), and groups identical `\1c&HBBGGRR&` rectangles into minimal ASS Dialogue events.

In [ ]:
#@title 5. Build Optimized Frame-by-Frame ASS \p1 Vector Mesh { display-mode: "form" }
import math
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple
import cv2
import numpy as np

GRID_PATCH_SIZE_PX = 4  #@param {type:"slider", min:1, max:16, step:1}
COLOR_MERGE_TOLERANCE = 6  #@param {type:"slider", min:0, max:30, step:1}
EDGE_BLUR = 0.6  #@param {type:"number"}
FRAME_STEP = 1  #@param {type:"slider", min:1, max:5, step:1}  # 1 = every frame, 2 = every 2nd frame
BASE_LAYER = 0  #@param {type:"integer"}
STYLE_NAME = "Default"  #@param {type:"string"}

MAX_RECTS_PER_EVENT = 1500


@dataclass
class RectPatch:
    x1: int
    y1: int
    x2: int
    y2: int
    bgr: Tuple[float, float, float]
    area: int


def format_ass_num(val: float) -> str:
    if abs(val - round(val)) < 1e-6:
        return str(int(round(val)))
    return f"{val:.3f}".rstrip("0").rstrip(".")


def bgr_to_ass_hex(bgr: Tuple[int, int, int]) -> str:
    b = max(0, min(255, int(bgr[0])))
    g = max(0, min(255, int(bgr[1])))
    r = max(0, min(255, int(bgr[2])))
    return f"&H{b:02X}{g:02X}{r:02X}&"


def quantize_rect_colors(rects: List[RectPatch], tolerance: int) -> Dict[Tuple[int, int, int], List[RectPatch]]:
    if not rects:
        return {}
    if tolerance <= 0:
        groups: Dict[Tuple[int, int, int], List[RectPatch]] = defaultdict(list)
        for rect in rects:
            key = (
                max(0, min(255, int(round(rect.bgr[0])))),
                max(0, min(255, int(round(rect.bgr[1])))),
                max(0, min(255, int(round(rect.bgr[2])))),
            )
            groups[key].append(rect)
        return groups

    snap_thresh = max(1.0, float(tolerance) * 0.35)
    indexed = sorted(enumerate(rects), key=lambda item: item[1].area, reverse=True)
    clusters = []
    assignments = {}

    for orig_idx, rect in indexed:
        rb, rg, rr = rect.bgr
        best_c = -1
        best_d = snap_thresh + 1.0
        for c_idx, cl in enumerate(clusters):
            cb, cg, cr = cl["mean"]
            dist = math.sqrt((rb - cb) ** 2 + (rg - cg) ** 2 + (rr - cr) ** 2)
            if dist <= snap_thresh and dist < best_d:
                best_d = dist
                best_c = c_idx
        if best_c < 0:
            clusters.append({
                "sb": rb * rect.area, "sg": rg * rect.area, "sr": rr * rect.area,
                "w": float(rect.area), "mean": (rb, rg, rr),
            })
            best_c = len(clusters) - 1
        else:
            cl = clusters[best_c]
            cl["sb"] += rb * rect.area
            cl["sg"] += rg * rect.area
            cl["sr"] += rr * rect.area
            cl["w"] += float(rect.area)
            wt = cl["w"]
            cl["mean"] = (cl["sb"] / wt, cl["sg"] / wt, cl["sr"] / wt)
        assignments[orig_idx] = best_c

    groups = defaultdict(list)
    for orig_idx, rect in enumerate(rects):
        mb, mg, mr = clusters[assignments[orig_idx]]["mean"]
        key = (
            max(0, min(255, int(round(mb)))),
            max(0, min(255, int(round(mg)))),
            max(0, min(255, int(round(mr)))),
        )
        groups[key].append(rect)
    return groups


def build_frame_vector_mesh(
    inpainted_crop: np.ndarray,
    crop_box: Tuple[int, int, int, int],
    orig_mask_full: np.ndarray,
    grid_size: int,
    tolerance: int,
    vid_w: int,
    vid_h: int,
    script_w: int,
    script_h: int,
) -> Tuple[Dict[Tuple[int, int, int], List[RectPatch]], int, int]:
    crop_x0, crop_y0, _, _ = crop_box
    g = max(1, min(16, int(grid_size)))
    tol = max(0, min(30, int(tolerance)))

    coverage = cv2.dilate(orig_mask_full, np.ones((3, 3), dtype=np.uint8), iterations=1)
    ys, xs = np.nonzero(coverage)
    if len(xs) == 0:
        return {}, 0, 0

    bx0, bx1 = int(xs.min()), int(xs.max()) + 1
    by0, by1 = int(ys.min()), int(ys.max()) + 1
    rows = int(math.ceil((by1 - by0) / float(g)))
    cols = int(math.ceil((bx1 - bx0) / float(g)))

    active = np.zeros((rows, cols), dtype=bool)
    cell_colors = np.zeros((rows, cols, 3), dtype=np.float32)
    cell_weights = np.zeros((rows, cols), dtype=np.float32)
    active_cells = 0

    for r in range(rows):
        y_start = by0 + r * g
        y_end = min(by1, y_start + g)
        cy_start, cy_end = y_start - crop_y0, y_end - crop_y0
        for c in range(cols):
            x_start = bx0 + c * g
            x_end = min(bx1, x_start + g)
            if not np.any(coverage[y_start:y_end, x_start:x_end]):
                continue
            patch = inpainted_crop[cy_start:cy_end, x_start - crop_x0 : x_end - crop_x0]
            if patch.size == 0:
                continue
            active[r, c] = True
            cell_colors[r, c] = patch.mean(axis=(0, 1))
            cell_weights[r, c] = float((y_end - y_start) * (x_end - x_start))
            active_cells += 1

    visited = ~active
    tol_sq = float(tol) * float(tol)
    sx = float(script_w) / float(vid_w) if vid_w > 0 else 1.0
    sy = float(script_h) / float(vid_h) if vid_h > 0 else 1.0
    merged: List[RectPatch] = []

    def can_merge(rr: int, cc: int, sb: float, sg: float, sr: float) -> bool:
        if visited[rr, cc]:
            return False
        db = float(cell_colors[rr, cc, 0]) - sb
        dg = float(cell_colors[rr, cc, 1]) - sg
        dr = float(cell_colors[rr, cc, 2]) - sr
        return (db * db + dg * dg + dr * dr) <= tol_sq

    for r in range(rows):
        for c in range(cols):
            if visited[r, c]:
                continue
            sb, sg, sr = float(cell_colors[r, c, 0]), float(cell_colors[r, c, 1]), float(cell_colors[r, c, 2])

            w1 = 1
            while c + w1 < cols and can_merge(r, c + w1, sb, sg, sr):
                w1 += 1
            h1 = 1
            while r + h1 < rows and all(can_merge(r + h1, cc, sb, sg, sr) for cc in range(c, c + w1)):
                h1 += 1

            h2 = 1
            while r + h2 < rows and can_merge(r + h2, c, sb, sg, sr):
                h2 += 1
            w2 = 1
            while c + w2 < cols and all(can_merge(rr, c + w2, sb, sg, sr) for rr in range(r, r + h2)):
                w2 += 1

            bw, bh = (w1, h1) if (w1 * h1) >= (w2 * h2) else (w2, h2)
            visited[r : r + bh, c : c + bw] = True

            sub_w = cell_weights[r : r + bh, c : c + bw]
            w_sum = float(sub_w.sum())
            if w_sum > 0:
                sub_c = cell_colors[r : r + bh, c : c + bw]
                ab = float((sub_c[:, :, 0] * sub_w).sum() / w_sum)
                ag = float((sub_c[:, :, 1] * sub_w).sum() / w_sum)
                ar = float((sub_c[:, :, 2] * sub_w).sum() / w_sum)
            else:
                ab, ag, ar = sb, sg, sr

            px1, py1 = bx0 + c * g, by0 + r * g
            px2, py2 = min(vid_w, bx0 + (c + bw) * g), min(vid_h, by0 + (r + bh) * g)
            sx1, sy1 = int(round(px1 * sx)), int(round(py1 * sy))
            sx2, sy2 = max(int(round(px2 * sx)), int(round(px1 * sx)) + 1), max(int(round(py2 * sy)), int(round(py1 * sy)) + 1)

            merged.append(RectPatch(sx1, sy1, sx2, sy2, (ab, ag, ar), (sx2 - sx1) * (sy2 - sy1)))

    return quantize_rect_colors(merged, tol), active_cells, len(merged)


blur_str = format_ass_num(max(0.0, float(EDGE_BLUR)))
step = max(1, int(FRAME_STEP))
style_clean = (STYLE_NAME or "Default").strip() or "Default"

GENERATED_DIALOGUE_LINES: List[str] = []
total_cells = 0
total_rects = 0
n_seq = len(INPAINTED_CROPS)

for idx in range(0, n_seq, step):
    frame_clip = PER_FRAME_CLIPS[idx]
    if not frame_clip or not np.any(ORIG_MASKS[idx]):
        continue

    f_start = eff_start_frame + idx
    f_end = min(eff_start_frame + n_seq, f_start + step)
    t_start_ms = OFFSET_MS + round((f_start * 1000.0) / VIDEO_FPS)
    t_end_ms = OFFSET_MS + round((f_end * 1000.0) / VIDEO_FPS)
    start_ass = ms_to_ass_time(t_start_ms)
    end_ass = ms_to_ass_time(t_end_ms)

    color_groups, n_cells, n_rects = build_frame_vector_mesh(
        INPAINTED_CROPS[idx],
        CROP_BOX,
        ORIG_MASKS[idx],
        GRID_PATCH_SIZE_PX,
        COLOR_MERGE_TOLERANCE,
        VIDEO_WIDTH,
        VIDEO_HEIGHT,
        EFF_PLAY_RES_X,
        EFF_PLAY_RES_Y,
    )
    total_cells += n_cells
    total_rects += n_rects

    sorted_groups = sorted(color_groups.items(), key=lambda it: sum(r.area for r in it[1]), reverse=True)
    layer_offset = 0

    for bgr_key, rects in sorted_groups:
        ass_color = bgr_to_ass_hex(bgr_key)
        rects_sorted = sorted(rects, key=lambda r: (r.y1, r.x1))
        for c_start in range(0, len(rects_sorted), MAX_RECTS_PER_EVENT):
            chunk = rects_sorted[c_start : c_start + MAX_RECTS_PER_EVENT]
            draw_cmds = " ".join(
                f"m {r.x1} {r.y1} l {r.x2} {r.y1} {r.x2} {r.y2} {r.x1} {r.y2}"
                for r in chunk
            )
            layer = int(BASE_LAYER) + layer_offset
            layer_offset += 1
            text = (
                f"{{\\an7\\pos(0,0)\\bord0\\shad0\\blur{blur_str}"
                f"\\1c{ass_color}\\clip({frame_clip})\\p1}}{draw_cmds}"
            )
            GENERATED_DIALOGUE_LINES.append(
                f"Dialogue: {layer},{start_ass},{end_ass},{style_clean},inpaint-mesh,0,0,0,,{text}"
            )

reduction_pct = (1.0 - (total_rects / float(total_cells))) * 100.0 if total_cells > 0 else 0.0
print(f"Processed Keyframes : {len(range(0, n_seq, step))} (step={step})")
print(f"Raw Grid Cells      : {total_cells}")
print(f"Merged Rectangles   : {total_rects} ({reduction_pct:.1f}% reduction via 2D greedy merge)")
print(f"Total ASS Events    : {len(GENERATED_DIALOGUE_LINES)}")

## 6. Export & Download `rebuilt_motion_patch.ass`

Saves the complete ASS subtitle file (with `[Script Info]` matching your video's `PlayResX`/`PlayResY`, `[V4+ Styles]`, and `[Events]`) and starts a browser download in Google Colab.

In [ ]:
#@title 6. Export rebuilt_motion_patch.ass & Download { display-mode: "form" }
OUTPUT_ASS_FILENAME = "rebuilt_motion_patch.ass"  #@param {type:"string"}

overall_start_ms = OFFSET_MS + round((eff_start_frame * 1000.0) / VIDEO_FPS)
overall_end_ms = OFFSET_MS + round(((eff_end_frame + 1) * 1000.0) / VIDEO_FPS)
overall_start_ass = ms_to_ass_time(overall_start_ms)
overall_end_ass = ms_to_ass_time(overall_end_ms)

style_clean = (STYLE_NAME or "Default").strip() or "Default"
styles_block = [
    "Style: Default,Arial,20,&H00FFFFFF,&H000000FF,&H00000000,&H00000000,0,0,0,0,100,100,0,0,1,0,0,7,0,0,0,1"
]
if style_clean != "Default":
    styles_block.append(
        f"Style: {style_clean},Arial,20,&H00FFFFFF,&H000000FF,&H00000000,&H00000000,0,0,0,0,100,100,0,0,1,0,0,7,0,0,0,1"
    )

guide_note = (
    "SAM 2 tracked guide clip (first active frame)"
    if globals().get("IS_MOVING_OBJECT_MODE", False)
    else "Original guide clip"
)

ass_content = "\n".join(
    [
        "[Script Info]",
        "; Script generated by ass-inpaint-mesh-motion.ipynb",
        "Title: ASS Inpaint Motion Mesh",
        "ScriptType: v4.00+",
        "WrapStyle: 0",
        "ScaledBorderAndShadow: yes",
        "YCbCr Matrix: TV.709",
        f"PlayResX: {EFF_PLAY_RES_X}",
        f"PlayResY: {EFF_PLAY_RES_Y}",
        "",
        "[V4+ Styles]",
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding",
        *styles_block,
        "",
        "[Events]",
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text",
        f"Comment: 0,{overall_start_ass},{overall_end_ass},{style_clean},guide,0,0,0,,{{\\clip({RAW_CLIP_INNER})}}{guide_note}",
        *GENERATED_DIALOGUE_LINES,
        "",
    ]
)

with open(OUTPUT_ASS_FILENAME, "w", encoding="utf-8-sig") as fh:
    fh.write(ass_content)

size_kb = os.path.getsize(OUTPUT_ASS_FILENAME) / 1024.0
print(f"Saved {OUTPUT_ASS_FILENAME} ({size_kb:.1f} KB, {len(GENERATED_DIALOGUE_LINES)} Dialogue lines).")

try:
    from google.colab import files
    files.download(OUTPUT_ASS_FILENAME)
except ImportError:
    print(f"Not running in Google Colab; file saved locally at {os.path.abspath(OUTPUT_ASS_FILENAME)}")